# v6 — Frozen DINOv3 ViT-H+/16 pada data mentah resmi (fold 0)

**Pertanyaan screen:** apakah checkpoint publik-gated `facebook/dinov3-vith16plus-pretrain-lvd1689m` cukup menjanjikan pada fold 0 untuk layak dipertimbangkan dalam eksperimen H+ terpisah yang lebih lengkap?

Notebook ini adalah **screen train-only yang hemat komputasi** di Kaggle GPU T4 ×2. Data mentah, fold, resolusi 384×384, FP32 backbone, representasi CLS + rerata true-patch, arsitektur head MLP, dan evaluasi original-view mengikuti arah v5. Namun, ini **bukan perbandingan protocol-matched**: completed v5 full run melatih probe dengan original+hflip dan seed 42+314, sedangkan v6 sengaja memakai original-only dan seed 42 saja.

> Tidak ada akses isi test, inferensi test, pembuatan submission, full five-fold, pencarian resep, TTA, ensemble, relabeling, atau fine-tuning di notebook ini.

## Landasan, kepatuhan, dan perbandingan yang sah

- Metrik kompetisi adalah macro F1 untuk tiga kelas: `0=Recyclable`, `1=Electronic`, `2=Organic`.
- Sumber pengembangan kanonik adalah label train mentah resmi: 9.999 / 3.961 / 12.567 gambar (total 26.527). Tidak ada data berlabel eksternal, metadata nonvisual, pseudo-label, relabel manual, atau penghapusan duplikat.
- Checkpoint H+ dilatih pada web/LVD-1689M, bukan citra satelit. Bobot bersifat gated: lisensi model harus diterima pada akun Hugging Face yang sama dengan pemilik token.
- **V5 ViT-L frozen raw full-run fold-0 original-view macro F1 = 0,985022** dipakai sebagai referensi proyek, bukan baseline kausal atau protocol-matched. Completed v5 full run melatih probe dengan feature views original+hflip dan merata-ratakan dua head seed (42, 314); v6 menjalankan tepat satu head, original-only, seed 42.
- Konteks full-OOF original-view v5: F1 Recyclable 0,981044; Electronic 0,991683; Organic 0,985721; confusion Recyclable→Organic 141 dan Organic→Recyclable 195. Angka ini tidak dipakai sebagai pembanding fold-0 v6.
- Official test berisi 1.458 gambar, tetapi isi, template, dan prediksinya tetap tidak disentuh karena v6 hanya model screening berbasis train.
- Pada v5, kombinasi probabilitas inferensi original+hflip meningkatkan full-OOF macro F1 dari 0,986149 menjadi 0,986239 (sekitar 0,000090), di bawah ambang proyek 0,0002, sehingga final inference view yang dipilih adalah original-only. Ini tidak membuktikan bahwa hflip sebagai feature augmentation saat melatih probe tidak berguna.
- Kontrak final v6 mencocokkan budget full-probe v5 yang telah selesai: maksimal 15 epoch, LR MLP `8e-4`, weight decay `1e-4`, patience 5, dan clip norm 1,0. V6 hanya menjalankan seed 42; seed 314 tetap metadata historis v5 dan tidak disiapkan sebagai run tersembunyi.

Selisih numerik terhadap 0,985022 hanyalah `reference_delta_vs_v5_fullrun_fold0`; nilainya tidak boleh ditafsirkan sebagai efek murni skala backbone. Ambang akhir adalah aturan keputusan proyek, bukan bukti kausal, dan notebook tidak otomatis memulai eksperimen lanjutan.

## 1. Setup Kaggle

Aktifkan internet dan accelerator **GPU T4 ×2**. Cell instalasi dibatasi pada lini Transformers yang sudah memiliki dukungan DINOv3; tidak ada klaim bahwa versi tertentu telah diverifikasi di luar runtime Kaggle ini.

In [ ]:
# DINOv3 tersedia pada rilis Transformers >= 4.56; batas mayor mencegah perubahan API tak terkendali.
!pip install -q --upgrade "transformers>=4.56,<5"


In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageFile, ImageOps
from huggingface_hub import model_info
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision.transforms import InterpolationMode, v2
from tqdm.auto import tqdm
import transformers
from transformers import AutoImageProcessor, AutoModel

NOTEBOOK_STARTED_AT = time.perf_counter()
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)

EXPERIMENT_NAME = 'v6_dinov3_vithplus_frozen_raw_fold0'
EXPERIMENT_PURPOSE = 'compute-efficient H+ fold-0 screen against a non-protocol-matched v5 project reference'
V6_REVISION_MARKER = 'final_v6_epoch15_seed42_patterned_preflight_v3'
MODEL_ID = 'facebook/dinov3-vith16plus-pretrain-lvd1689m'
MODEL_REVISION_REQUEST = 'main'
EXPECTED_HPLUS_HIDDEN_SIZE = 1280
IMAGE_SIZE = 384
FEATURE_VIEW = 'orig'
POOLING_REPRESENTATION = 'concat(cls, mean_true_patch_tokens)'
BACKBONE_COMPUTE_DTYPE = torch.float32
BACKBONE_COMPUTE_DTYPE_NAME = 'float32'
FEATURE_STORAGE_DTYPE = torch.float16
FEATURE_STORAGE_DTYPE_NAME = 'float16'
ATTENTION_IMPLEMENTATION = 'sdpa'

SEED = 42
N_FOLDS = 5
VALIDATION_FOLD = 0
NUM_CLASSES = 3
FEATURE_BATCH_SIZE = 2       # total DataParallel batch: satu gambar per T4
FEATURE_WORKERS = 2
FEATURE_SHARD_SIZE = 512
HEAD_BATCH_SIZE = 512
PREDICT_BATCH_SIZE = 1024
MAX_EPOCHS = 15              # matched to completed v5 FULL_EPOCHS=15
EARLY_STOPPING_PATIENCE = 5
HEAD_LEARNING_RATE = 8e-4
HEAD_WEIGHT_DECAY = 1e-4
GRAD_CLIP_MAX_NORM = 1.0
PROBE_RECIPE_NAME = 'mlp_concat_weighted_ce'
V5_REFERENCE_FOLD0_MACRO_F1 = 0.985022
GO_THRESHOLD = 0.9880
CONDITIONAL_THRESHOLD = 0.9865

LARGE_IMAGE_MP_THRESHOLD = 8_000_000
LARGE_IMAGE_MAX_SOURCE_SIDE = 4096
SAFE_COPY_MAX_SIDE = 1024
SAFE_COPY_JPEG_QUALITY = 95
PIL_MAX_IMAGE_PIXELS = 200_000_000
CACHE_SCHEMA_VERSION = 8
PREFLIGHT_SCHEMA_VERSION = 2

CLASS_DIR_TO_LABEL = {
    '0_Recyclable': 0,
    '1_Electronic': 1,
    '2_Organic': 2,
}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
EXPECTED_RAW_COUNTS = {0: 9999, 1: 3961, 2: 12567}
EXPECTED_TRAIN_ROWS = 26527
ALLOWED_IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

assert MODEL_ID == 'facebook/dinov3-vith16plus-pretrain-lvd1689m'
assert FEATURE_VIEW == 'orig' and VALIDATION_FOLD == 0 and SEED == 42
assert BACKBONE_COMPUTE_DTYPE is torch.float32
assert FEATURE_STORAGE_DTYPE is torch.float16
assert FEATURE_BATCH_SIZE == 2
assert PROBE_RECIPE_NAME == 'mlp_concat_weighted_ce'
assert MAX_EPOCHS == 15

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')
Image.MAX_IMAGE_PIXELS = PIL_MAX_IMAGE_PIXELS
ImageFile.LOAD_TRUNCATED_IMAGES = False

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError('Notebook v6 ini hanya ditujukan untuk runtime Kaggle dengan /kaggle/working.')
ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
CACHE_DIR = ARTIFACT_DIR / 'feature_cache'
SHARDS_DIR = CACHE_DIR / 'shards'
for directory in (ARTIFACT_DIR, CACHE_DIR, SHARDS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def sha256_json(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def json_default(value):
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Not JSON serializable: {type(value).__name__}')

def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, default=json_default)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def write_csv_atomic(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)

def atomic_torch_save(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('wb') as handle:
        torch.save(payload, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

print('Artifacts v6:', ARTIFACT_DIR)


## 2. Kontrak data mentah dan fold

Cell di bawah hanya mendefinisikan resolver dan guard data. Enumerasi tree `train`, assignment fold, audit dimensi, dan safe-copy baru dijalankan setelah preflight sintetis berhasil. Resolver tidak pernah turun ke direktori bernama `test`; hanya satu root dengan signature tiga kelas dan hitungan persis yang boleh cocok.

Urutan baris mengikuti v5: relative image ID disortir, lalu `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`. Assignment disimpan setelah preflight dan sebelum decoding citra train.

In [ ]:
def discover_named_train_dirs(base, max_depth=3):
    if not base.is_dir():
        return []
    found = []
    stack = [(base, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            name = child.name.lower()
            if name == 'test':
                continue
            if name == 'train':
                found.append(child)
            elif depth < max_depth:
                stack.append((child, depth + 1))
    return found

def exact_raw_train_signature(train_dir):
    if not train_dir.is_dir():
        return None
    try:
        root_entries = list(train_dir.iterdir())
        if (
            {path.name for path in root_entries} != set(CLASS_DIR_TO_LABEL)
            or any(path.is_symlink() or not path.is_dir() for path in root_entries)
        ):
            return None
        counts = {}
        for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
            entries = list((train_dir / class_dir_name).iterdir())
            if any(
                entry.is_symlink()
                or not entry.is_file()
                or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
                for entry in entries
            ):
                return None
            counts[label] = len(entries)
        return counts if counts == EXPECTED_RAW_COUNTS else None
    except OSError:
        return None

def resolve_train_dir():
    candidates = [Path.cwd() / 'data' / 'train', KAGGLE_WORKING / 'data' / 'train']
    candidates.extend(discover_named_train_dirs(Path('/kaggle/input'), max_depth=3))
    matches = {
        candidate.resolve()
        for candidate in candidates
        if exact_raw_train_signature(candidate) == EXPECTED_RAW_COUNTS
    }
    if len(matches) != 1:
        rendered = '\n'.join(f'  - {path}' for path in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError(
            'Harus ada tepat satu direktori train mentah resmi dengan signature '
            f'{EXPECTED_RAW_COUNTS}; ditemukan {len(matches)}:\n{rendered}. '
            'Periksa mount Kaggle dan singkirkan tree cleaned/augmented yang ambigu.'
        )
    return matches.pop()

def index_raw_train(train_dir):
    root_entries = list(train_dir.iterdir())
    actual_class_dirs = {path.name for path in root_entries}
    if (
        actual_class_dirs != set(CLASS_DIR_TO_LABEL)
        or any(path.is_symlink() or not path.is_dir() for path in root_entries)
    ):
        raise RuntimeError(
            f'Direktori kelas tidak cocok: {sorted(actual_class_dirs)}; '
            f'expected {sorted(CLASS_DIR_TO_LABEL)}.'
        )
    rows = []
    observed_counts = {}
    for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
        class_dir = train_dir / class_dir_name
        entries = sorted(class_dir.iterdir(), key=lambda path: path.name)
        invalid = [
            entry.name for entry in entries
            if entry.is_symlink()
            or not entry.is_file()
            or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
        ]
        if invalid:
            raise RuntimeError(f'Entry nested/unsupported/symlink di {class_dir}: {invalid[:10]}')
        observed_counts[label] = len(entries)
        for path in entries:
            relative_path = path.relative_to(train_dir).as_posix()
            rows.append({
                'image_id': relative_path,
                'relative_path': relative_path,
                'class_name': CLASS_NAMES[label],
                'label': label,
                'path': str(path.resolve()),
            })
    if observed_counts != EXPECTED_RAW_COUNTS:
        raise RuntimeError(
            f'Signature raw salah: observed={observed_counts}, expected={EXPECTED_RAW_COUNTS}. '
            'V6 menolak label cleaned/relabeled/augmented.'
        )
    frame = pd.DataFrame(rows).sort_values('image_id').reset_index(drop=True)
    if len(frame) != EXPECTED_TRAIN_ROWS or not frame['image_id'].is_unique:
        raise RuntimeError('Jumlah baris atau keunikan image_id train tidak memenuhi kontrak raw.')
    if set(frame['label'].unique()) != set(range(NUM_CLASSES)):
        raise RuntimeError('Tidak semua tiga kelas hadir pada train mentah.')
    return frame

def inspect_image_dimensions(path):
    try:
        with Image.open(path) as image:
            width, height = image.size
        if width <= 0 or height <= 0:
            raise ValueError(f'invalid dimensions {width}x{height}')
        return int(width), int(height)
    except Exception as exc:
        raise RuntimeError(f'Gagal membaca header/dimensi citra train: {path}') from exc

def safe_copy_is_valid(path):
    try:
        with Image.open(path) as image:
            width, height = image.size
            mode = image.mode
            image.verify()
        return width > 0 and height > 0 and max(width, height) <= SAFE_COPY_MAX_SIDE and mode == 'RGB'
    except Exception:
        return False

def write_safe_copy(source_path, destination_path):
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination_path.with_name(
        f'{destination_path.stem}.tmp{destination_path.suffix.lower()}'
    )
    format_by_suffix = {
        '.jpg': 'JPEG', '.jpeg': 'JPEG', '.png': 'PNG',
        '.bmp': 'BMP', '.webp': 'WEBP',
    }
    image_format = format_by_suffix.get(destination_path.suffix.lower())
    if image_format is None:
        raise RuntimeError(f'Format safe-copy tidak didukung: {destination_path.suffix}')
    try:
        with Image.open(source_path) as image:
            image.draft('RGB', (SAFE_COPY_MAX_SIDE, SAFE_COPY_MAX_SIDE))
            image = ImageOps.exif_transpose(image).convert('RGB')
            image.thumbnail((SAFE_COPY_MAX_SIDE, SAFE_COPY_MAX_SIDE), Image.Resampling.LANCZOS)
            save_kwargs = {'quality': SAFE_COPY_JPEG_QUALITY} if image_format in {'JPEG', 'WEBP'} else {}
            image.save(temporary_path, format=image_format, **save_kwargs)
        os.replace(temporary_path, destination_path)
    except Exception as exc:
        raise RuntimeError(f'Gagal membuat safe-copy {destination_path} dari {source_path}') from exc
    if not safe_copy_is_valid(destination_path):
        raise RuntimeError(f'Safe-copy tidak valid setelah ditulis: {destination_path}')

def prepare_safe_image_paths(frame):
    prepared = frame.copy()
    prepared['source_path'] = prepared['path'].astype(str)
    original_ids = prepared['image_id'].astype(str).tolist()
    safe_root = ARTIFACT_DIR / 'safe_images' / (
        f'max_side_{SAFE_COPY_MAX_SIDE}_quality_{SAFE_COPY_JPEG_QUALITY}'
    ) / 'train'
    records = []
    created = reused = 0
    for index, row in tqdm(prepared.iterrows(), total=len(prepared), desc='image safety train'):
        source_path = Path(row['source_path'])
        if not source_path.is_file():
            raise RuntimeError(f'File sumber train tidak ditemukan: {source_path}')
        width, height = inspect_image_dimensions(source_path)
        needs_safe_copy = (
            width * height >= LARGE_IMAGE_MP_THRESHOLD
            or max(width, height) > LARGE_IMAGE_MAX_SOURCE_SIDE
        )
        effective_path = source_path
        if needs_safe_copy:
            destination_path = safe_root / Path(row['relative_path'])
            if safe_copy_is_valid(destination_path):
                reused += 1
            else:
                write_safe_copy(source_path, destination_path)
                created += 1
            effective_path = destination_path
        prepared.at[index, 'path'] = str(effective_path)
        records.append({
            'image_id': str(row['image_id']),
            'source_path': str(source_path),
            'effective_path': str(effective_path),
            'source_width': width,
            'source_height': height,
            'used_safe_copy': bool(needs_safe_copy),
        })
    if prepared['image_id'].astype(str).tolist() != original_ids:
        raise RuntimeError('Image safety mengubah urutan image_id.')
    missing = [path for path in prepared['path'] if not Path(path).is_file()]
    if missing:
        raise RuntimeError(f'{len(missing)} effective train paths hilang; contoh: {missing[:3]}')
    write_csv_atomic(pd.DataFrame(records), ARTIFACT_DIR / 'safe_image_map_train.csv')
    print(
        f'Image safety: {sum(record["used_safe_copy"] for record in records)} large, '
        f'{created} dibuat, {reused} dipakai ulang. Dimensi tidak menjadi fitur.'
    )
    return prepared

def load_rgb_image(path):
    try:
        with Image.open(path) as image:
            image.draft('RGB', (SAFE_COPY_MAX_SIDE, SAFE_COPY_MAX_SIDE))
            image = ImageOps.exif_transpose(image)
            return image.convert('RGB').copy()
    except Exception as exc:
        raise RuntimeError(f'Citra train korup/tidak dapat didekode: {path}') from exc


## 3. T4 ×2, akses gated, processor resmi, dan frozen H+

Terima lisensi checkpoint pada Hugging Face menggunakan akun pemilik token. Token dibaca dari environment lebih dahulu, lalu Kaggle Secret `HF_TOKEN`; nilainya tidak pernah dicetak, di-hash, atau disimpan.

Checkpoint ini diharapkan sekitar 840 juta parameter, hidden size 1280, empat register token, dan patch 16; nilai runtime tetap diturunkan dari config dan jumlah parameter aktual, bukan dipercaya dari angka dokumentasi.

Notebook menyelesaikan commit immutable sebelum memuat processor dan model, lalu memakai commit yang sama untuk keduanya. Semua guard gagal tertutup: tepat dua T4, ID/revision checkpoint, hidden size H+, patch 16, RGB, metadata register, sequence length, frozen state, dan FP32 parameter/buffer.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('CUDA tidak tersedia; V6 hanya boleh dijalankan pada Kaggle T4 x2.')
cuda_count = torch.cuda.device_count()
if cuda_count != 2:
    raise RuntimeError(
        f'V6 memerlukan tepat Kaggle GPU T4 x2; runtime melihat {cuda_count} CUDA device. '
        'Aktifkan accelerator dual-T4 dan mulai ulang sesi.'
    )
DEVICE_IDS = [0, 1]
GPU_RUNTIME_INFO = []
for device_id in DEVICE_IDS:
    properties = torch.cuda.get_device_properties(device_id)
    if 'T4' not in properties.name.upper():
        raise RuntimeError(
            f'cuda:{device_id} adalah {properties.name}, bukan T4. '
            'V6 dikunci untuk observasi kelayakan T4 x2.'
        )
    row = {
        'device_id': device_id,
        'name': properties.name,
        'total_vram_gib': properties.total_memory / 2**30,
    }
    GPU_RUNTIME_INFO.append(row)
    print(f"cuda:{device_id}: {row['name']}, {row['total_vram_gib']:.2f} GiB")

HF_TOKEN = os.environ.get('HF_TOKEN')
token_source = 'environment variable' if HF_TOKEN else None
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        token_source = 'private Kaggle Secret HF_TOKEN'
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError(
        'Checkpoint DINOv3 H+ bersifat gated. Terima lisensinya dengan akun yang sama, '
        'lalu tambahkan Kaggle Secret privat bernama HF_TOKEN. Token tidak akan disimpan.'
    )
print(f'Hugging Face authentication: {token_source}; token disembunyikan.')

try:
    RESOLVED_REVISION = model_info(
        MODEL_ID, revision=MODEL_REVISION_REQUEST, token=HF_TOKEN
    ).sha
    if (
        not RESOLVED_REVISION
        or len(RESOLVED_REVISION) != 40
        or any(character not in '0123456789abcdef' for character in RESOLVED_REVISION.lower())
    ):
        raise RuntimeError(f'Hub tidak mengembalikan commit SHA immutable: {RESOLVED_REVISION!r}')
    processor = AutoImageProcessor.from_pretrained(
        MODEL_ID, revision=RESOLVED_REVISION, token=HF_TOKEN
    )
    backbone = AutoModel.from_pretrained(
        MODEL_ID,
        revision=RESOLVED_REVISION,
        token=HF_TOKEN,
        dtype=BACKBONE_COMPUTE_DTYPE,
        attn_implementation=ATTENTION_IMPLEMENTATION,
    )
except Exception as exc:
    raise RuntimeError(
        f'Gagal mengakses {MODEL_ID}. Periksa internet Kaggle, penerimaan lisensi, '
        'hak HF_TOKEN, commit Hub, serta kompatibilitas Transformers. Tidak ada fallback.'
    ) from exc

del HF_TOKEN
config = backbone.config
loaded_name = str(getattr(config, '_name_or_path', '')).rstrip('/')
if loaded_name != MODEL_ID:
    raise RuntimeError(f'Silent checkpoint fallback: config name={loaded_name!r}, expected={MODEL_ID!r}.')
config_revision = getattr(config, '_commit_hash', None)
if config_revision != RESOLVED_REVISION:
    raise RuntimeError(
        f'Revision config {config_revision!r} berbeda dari commit Hub {RESOLVED_REVISION!r}.'
    )

required_config_fields = ('hidden_size', 'patch_size', 'num_register_tokens', 'num_channels')
missing_config_fields = [name for name in required_config_fields if getattr(config, name, None) is None]
if missing_config_fields:
    raise RuntimeError(f'Config H+ kehilangan metadata wajib: {missing_config_fields}; menolak menebak.')
HIDDEN_SIZE = int(config.hidden_size)
if HIDDEN_SIZE != EXPECTED_HPLUS_HIDDEN_SIZE:
    raise RuntimeError(
        f'Hidden size {HIDDEN_SIZE} tidak konsisten dengan H+ yang dipilih '
        f'(expected {EXPECTED_HPLUS_HIDDEN_SIZE}).'
    )
NUM_REGISTER_TOKENS = int(config.num_register_tokens)
if NUM_REGISTER_TOKENS < 0:
    raise RuntimeError(f'Jumlah register token tidak valid: {NUM_REGISTER_TOKENS}.')
PATCH_SIZE = config.patch_size
if isinstance(PATCH_SIZE, (list, tuple)):
    if len(PATCH_SIZE) != 2 or PATCH_SIZE[0] != PATCH_SIZE[1]:
        raise RuntimeError(f'Patch size non-square tidak didukung: {PATCH_SIZE}.')
    PATCH_SIZE = PATCH_SIZE[0]
PATCH_SIZE = int(PATCH_SIZE)
if PATCH_SIZE != 16:
    raise RuntimeError(f'Checkpoint H+/16 wajib patch size 16, got {PATCH_SIZE}.')
if IMAGE_SIZE % PATCH_SIZE:
    raise RuntimeError(f'IMAGE_SIZE={IMAGE_SIZE} tidak habis dibagi PATCH_SIZE={PATCH_SIZE}.')
if int(config.num_channels) != 3:
    raise RuntimeError(f'Input model wajib RGB tiga-channel, got num_channels={config.num_channels}.')
EXPECTED_PATCH_TOKENS = (IMAGE_SIZE // PATCH_SIZE) ** 2
EXPECTED_SEQUENCE_TOKENS = 1 + NUM_REGISTER_TOKENS + EXPECTED_PATCH_TOKENS
PROBE_INPUT_DIM = 2 * HIDDEN_SIZE

backbone = backbone.to(dtype=BACKBONE_COMPUTE_DTYPE)
backbone.eval().requires_grad_(False)
parameter_dtypes = {parameter.dtype for parameter in backbone.parameters()}
floating_buffer_dtypes = {buffer.dtype for buffer in backbone.buffers() if buffer.is_floating_point()}
if parameter_dtypes != {BACKBONE_COMPUTE_DTYPE}:
    raise RuntimeError(f'Parameter backbone tidak seragam FP32: {parameter_dtypes}.')
if not floating_buffer_dtypes.issubset({BACKBONE_COMPUTE_DTYPE}):
    raise RuntimeError(f'Floating buffer backbone tidak seragam FP32: {floating_buffer_dtypes}.')
BACKBONE_PARAMETER_COUNT = sum(parameter.numel() for parameter in backbone.parameters())
BACKBONE_TRAINABLE_PARAMETER_COUNT = sum(
    parameter.numel() for parameter in backbone.parameters() if parameter.requires_grad
)
if BACKBONE_TRAINABLE_PARAMETER_COUNT != 0:
    raise RuntimeError('Backbone H+ tidak sepenuhnya frozen.')

PROCESSOR_DO_RESIZE = getattr(processor, 'do_resize', None)
PROCESSOR_DO_RESCALE = getattr(processor, 'do_rescale', None)
PROCESSOR_DO_NORMALIZE = getattr(processor, 'do_normalize', None)
PROCESSOR_DO_CONVERT_RGB = getattr(processor, 'do_convert_rgb', None)
PROCESSOR_DO_CENTER_CROP = getattr(processor, 'do_center_crop', None)
PROCESSOR_RESCALE_FACTOR = float(getattr(processor, 'rescale_factor', float('nan')))
PROCESSOR_MEAN = tuple(float(value) for value in getattr(processor, 'image_mean', ()))
PROCESSOR_STD = tuple(float(value) for value in getattr(processor, 'image_std', ()))
PROCESSOR_NATIVE_SIZE = dict(getattr(processor, 'size', {}) or {})
try:
    PROCESSOR_RESAMPLE = Image.Resampling(int(processor.resample))
except Exception as exc:
    raise RuntimeError(f'Resampling processor tidak dikenali: {getattr(processor, "resample", None)}') from exc
resample_to_interpolation = {
    Image.Resampling.NEAREST: InterpolationMode.NEAREST,
    Image.Resampling.BOX: InterpolationMode.BOX,
    Image.Resampling.BILINEAR: InterpolationMode.BILINEAR,
    Image.Resampling.HAMMING: InterpolationMode.HAMMING,
    Image.Resampling.BICUBIC: InterpolationMode.BICUBIC,
    Image.Resampling.LANCZOS: InterpolationMode.LANCZOS,
}
if PROCESSOR_RESAMPLE not in resample_to_interpolation:
    raise RuntimeError(f'Resampling processor belum dipetakan: {PROCESSOR_RESAMPLE}.')
PROCESSOR_INTERPOLATION = resample_to_interpolation[PROCESSOR_RESAMPLE]
if PROCESSOR_DO_RESIZE is not True:
    raise RuntimeError('Processor resmi harus mengaktifkan resize; hanya target spasial yang dioverride.')
if PROCESSOR_DO_RESCALE is not True or not np.isclose(PROCESSOR_RESCALE_FACTOR, 1.0 / 255.0):
    raise RuntimeError(f'Rescale processor wajib 1/255, got {PROCESSOR_RESCALE_FACTOR}.')
if PROCESSOR_DO_NORMALIZE is not True or len(PROCESSOR_MEAN) != 3 or len(PROCESSOR_STD) != 3:
    raise RuntimeError('Processor resmi wajib menyediakan normalisasi mean/std tiga channel.')
if not np.isfinite(PROCESSOR_MEAN).all() or not np.isfinite(PROCESSOR_STD).all() or min(PROCESSOR_STD) <= 0:
    raise RuntimeError('Mean/std processor tidak finite atau std tidak positif.')
if PROCESSOR_DO_CONVERT_RGB not in (None, True):
    raise RuntimeError('Processor tidak konsisten dengan loader RGB wajib.')
if PROCESSOR_DO_CENTER_CROP is True:
    raise RuntimeError('Processor meminta center crop; eksperimen ini dikunci direct square resize.')

PREPROCESS_CONFIG = {
    'model_id': MODEL_ID,
    'model_revision': RESOLVED_REVISION,
    'processor_class': type(processor).__name__,
    'processor_native_size': PROCESSOR_NATIVE_SIZE,
    'spatial_override': {'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
    'resize_geometry': 'direct_square',
    'view': FEATURE_VIEW,
    'do_resize': PROCESSOR_DO_RESIZE,
    'do_convert_rgb': PROCESSOR_DO_CONVERT_RGB,
    'implemented_rgb_conversion': True,
    'exif_transpose': True,
    'do_rescale': PROCESSOR_DO_RESCALE,
    'rescale_factor': PROCESSOR_RESCALE_FACTOR,
    'rescale_implementation': 'torchvision.ToDtype(float32, scale=True)',
    'do_normalize': PROCESSOR_DO_NORMALIZE,
    'image_mean': list(PROCESSOR_MEAN),
    'image_std': list(PROCESSOR_STD),
    'resample': PROCESSOR_RESAMPLE.name,
    'antialias': True,
    'large_image_mp_threshold': LARGE_IMAGE_MP_THRESHOLD,
    'large_image_max_source_side': LARGE_IMAGE_MAX_SOURCE_SIDE,
    'safe_copy_max_side': SAFE_COPY_MAX_SIDE,
    'safe_copy_jpeg_quality': SAFE_COPY_JPEG_QUALITY,
    'pil_max_image_pixels': PIL_MAX_IMAGE_PIXELS,
    'augmentation': 'none',
}
PREPROCESS_SIGNATURE = sha256_json(PREPROCESS_CONFIG)

print(f'Resolved immutable revision: {RESOLVED_REVISION}')
print(
    f'hidden={HIDDEN_SIZE}, patch={PATCH_SIZE}, registers={NUM_REGISTER_TOKENS}, '
    f'patch_tokens={EXPECTED_PATCH_TOKENS}, sequence_tokens={EXPECTED_SEQUENCE_TOKENS}'
)
print(f'Backbone parameters={BACKBONE_PARAMETER_COUNT:,}; trainable=0; compute=FP32')
print('Processor audit:', PREPROCESS_CONFIG)


In [ ]:
image_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize(
        (IMAGE_SIZE, IMAGE_SIZE), interpolation=PROCESSOR_INTERPOLATION, antialias=True
    ),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=PROCESSOR_MEAN, std=PROCESSOR_STD),
])

class OriginalViewTrainDataset(Dataset):
    def __init__(self, frame):
        self.paths = frame['path'].astype(str).tolist()
        self.labels = frame['label'].astype(int).tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = load_rgb_image(self.paths[index])
        return image_transform(image), self.labels[index], index

class FrozenDinoTokenPooler(nn.Module):
    def __init__(self, frozen_backbone):
        super().__init__()
        self.backbone = frozen_backbone

    def forward(self, pixel_values):
        if pixel_values.dtype != BACKBONE_COMPUTE_DTYPE:
            raise RuntimeError(f'Input backbone wajib FP32, got {pixel_values.dtype}.')
        output = self.backbone(pixel_values=pixel_values, return_dict=True)
        tokens = getattr(output, 'last_hidden_state', None)
        if tokens is None:
            raise RuntimeError('Model H+ tidak mengembalikan last_hidden_state.')
        if tokens.dtype != BACKBONE_COMPUTE_DTYPE:
            raise RuntimeError(f'Output backbone wajib FP32, got {tokens.dtype}.')
        if tokens.ndim != 3 or tokens.shape[1] != EXPECTED_SEQUENCE_TOKENS or tokens.shape[2] != HIDDEN_SIZE:
            raise RuntimeError(
                f'Shape token H+ tidak cocok: {tuple(tokens.shape)}; expected '
                f'(batch, {EXPECTED_SEQUENCE_TOKENS}, {HIDDEN_SIZE}).'
            )
        class_token = tokens[:, 0]
        patch_tokens = tokens[:, 1 + NUM_REGISTER_TOKENS:]
        if patch_tokens.shape[1] != EXPECTED_PATCH_TOKENS:
            raise RuntimeError('Eksklusi CLS + register menghasilkan jumlah true-patch yang salah.')
        patch_mean = patch_tokens.mean(dim=1)
        pooled = torch.stack((class_token, patch_mean), dim=1)
        expected_shape = (pixel_values.shape[0], 2, HIDDEN_SIZE)
        if tuple(pooled.shape) != expected_shape or pooled.dtype != BACKBONE_COMPUTE_DTYPE:
            raise RuntimeError(
                f'Kontrak pooled feature salah: shape={tuple(pooled.shape)}, dtype={pooled.dtype}; '
                f'expected={expected_shape}/float32.'
            )
        return pooled

pooler = FrozenDinoTokenPooler(backbone).to(device='cuda:0', dtype=BACKBONE_COMPUTE_DTYPE).eval()
pooler.requires_grad_(False)
if any(parameter.requires_grad for parameter in pooler.parameters()):
    raise RuntimeError('Pooler/backbone memiliki parameter trainable.')
if {parameter.dtype for parameter in pooler.parameters()} != {BACKBONE_COMPUTE_DTYPE}:
    raise RuntimeError('Base module DataParallel tidak seluruhnya FP32.')
feature_model = nn.DataParallel(pooler, device_ids=DEVICE_IDS, output_device=DEVICE_IDS[0]).eval()
if feature_model.training or feature_model.module.training:
    raise RuntimeError('DataParallel pooler/backbone wajib berada dalam evaluation mode.')
if any(parameter.requires_grad for parameter in feature_model.parameters()):
    raise RuntimeError('DataParallel backbone tidak sepenuhnya frozen.')
for device_id in DEVICE_IDS:
    torch.cuda.reset_peak_memory_stats(device_id)
print('Feature strategy: torch.nn.DataParallel pada cuda:0 dan cuda:1; total batch=2.')


## 4. Preflight minimal H+ FP32 pada T4 ×2

Sebelum enumerasi tree train dan audit 26.527 gambar, notebook membuat tepat dua citra RGB berpola gradien deterministik dengan dimensi sumber berbeda dan non-square (`517×293` serta `271×463`). Keduanya diproses menuju 384×384 oleh `AutoImageProcessor` resmi dengan spatial override yang sama, dibandingkan secara numerik dengan transform extraction manual, lalu melewati wrapper pooling dan jalur `DataParallel` yang sama dengan total batch 2.

Status preflight masih **pending** sampai cell ini dijalankan di Kaggle. Jika lolos, ia hanya membuktikan bahwa resize/normalisasi, model, layout token, pooling, dan satu forward minimal memakai jalur dua-T4 yang dikunci; ia **tidak** membuktikan bahwa extraction seluruh dataset akan muat atau selesai dalam batas waktu Kaggle.

In [ ]:
PREFLIGHT_PASSED = False
PREFLIGHT_REPORT_PATH = ARTIFACT_DIR / 'preflight_report.json'
preflight_source_specs = [
    {'width': 517, 'height': 293, 'pattern': 'xy_linear_gradients'},
    {'width': 271, 'height': 463, 'pattern': 'reversed_xy_linear_gradients'},
]
if (
    len(preflight_source_specs) != FEATURE_BATCH_SIZE
    or len({(spec['width'], spec['height']) for spec in preflight_source_specs}) != 2
    or any(
        spec['width'] <= 0
        or spec['height'] <= 0
        or spec['width'] == spec['height']
        or (spec['width'], spec['height']) == (IMAGE_SIZE, IMAGE_SIZE)
        for spec in preflight_source_specs
    )
):
    raise RuntimeError('Preflight wajib memakai tepat dua dimensi sumber non-square yang berbeda dari 384x384.')

preflight_images = []
for image_index, spec in enumerate(preflight_source_specs):
    x_axis = np.linspace(0.0, 1.0, spec['width'], dtype=np.float32)
    y_axis = np.linspace(0.0, 1.0, spec['height'], dtype=np.float32)
    x_grid, y_grid = np.meshgrid(x_axis, y_axis)
    if image_index == 0:
        channels = (x_grid, y_grid, 0.5 * (x_grid + y_grid))
    else:
        channels = (1.0 - y_grid, x_grid, (x_grid + 2.0 * y_grid) / 3.0)
    pattern = np.rint(255.0 * np.stack(channels, axis=-1)).astype(np.uint8)
    image = Image.fromarray(pattern)
    if image.mode != 'RGB' or image.size != (spec['width'], spec['height']):
        raise RuntimeError(f'Citra sintetis preflight tidak memenuhi kontrak RGB/dimensi: {image.mode}, {image.size}.')
    preflight_images.append(image)

official_preflight_batch = processor(
    images=preflight_images,
    return_tensors='pt',
    do_resize=True,
    size={'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
    do_rescale=True,
    do_normalize=True,
)
preflight_pixel_values = official_preflight_batch.get('pixel_values')
expected_input_shape = (FEATURE_BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
if not isinstance(preflight_pixel_values, torch.Tensor):
    raise RuntimeError('Processor resmi tidak menghasilkan tensor pixel_values pada preflight.')
if tuple(preflight_pixel_values.shape) != expected_input_shape:
    raise RuntimeError(
        f'Shape input preflight salah: {tuple(preflight_pixel_values.shape)}; '
        f'expected={expected_input_shape}.'
    )
if preflight_pixel_values.dtype != BACKBONE_COMPUTE_DTYPE:
    raise RuntimeError(f'Processor preflight wajib menghasilkan FP32, got {preflight_pixel_values.dtype}.')
if not torch.isfinite(preflight_pixel_values).all().item():
    raise RuntimeError('Processor resmi menghasilkan input preflight non-finite.')

manual_preflight_pixel_values = torch.stack([image_transform(image) for image in preflight_images])
if (
    tuple(manual_preflight_pixel_values.shape) != expected_input_shape
    or manual_preflight_pixel_values.dtype != BACKBONE_COMPUTE_DTYPE
    or not torch.isfinite(manual_preflight_pixel_values).all().item()
):
    raise RuntimeError('Transform extraction manual gagal kontrak shape/dtype/finite preflight.')
preflight_abs_difference = (
    manual_preflight_pixel_values - preflight_pixel_values
).abs()
preflight_transform_max_abs_tolerance = 0.05
preflight_transform_mean_abs_tolerance = 0.005
preflight_transform_max_abs_difference = float(preflight_abs_difference.max().item())
preflight_transform_mean_abs_difference = float(preflight_abs_difference.mean().item())
if (
    not torch.allclose(
        manual_preflight_pixel_values,
        preflight_pixel_values,
        rtol=0.0,
        atol=preflight_transform_max_abs_tolerance,
    )
    or preflight_transform_mean_abs_difference > preflight_transform_mean_abs_tolerance
):
    raise RuntimeError(
        'Transform extraction tidak konsisten dengan processor resmi pada resize non-square; '
        f'max_abs_diff={preflight_transform_max_abs_difference:.8f}, '
        f'mean_abs_diff={preflight_transform_mean_abs_difference:.8f}.'
    )

if cuda_count != 2 or DEVICE_IDS != [0, 1] or FEATURE_BATCH_SIZE != 2:
    raise RuntimeError('Preflight wajib tepat dua CUDA device dan total batch 2.')
if any('T4' not in row['name'].upper() for row in GPU_RUNTIME_INFO):
    raise RuntimeError('Preflight dikunci untuk dua NVIDIA T4.')
if any(parameter.requires_grad for parameter in feature_model.parameters()):
    raise RuntimeError('Backbone tidak frozen sebelum preflight.')
if {parameter.dtype for parameter in feature_model.parameters()} != {BACKBONE_COMPUTE_DTYPE}:
    raise RuntimeError('Parameter backbone tidak seluruhnya FP32 sebelum preflight.')
preflight_float_buffer_dtypes = {
    buffer.dtype for buffer in feature_model.buffers() if buffer.is_floating_point()
}
if not preflight_float_buffer_dtypes.issubset({BACKBONE_COMPUTE_DTYPE}):
    raise RuntimeError(f'Floating buffer bukan FP32 sebelum preflight: {preflight_float_buffer_dtypes}.')

preflight_allocated_before = {}
for device_id in DEVICE_IDS:
    torch.cuda.synchronize(device_id)
    preflight_allocated_before[device_id] = torch.cuda.memory_allocated(device_id)
    torch.cuda.reset_peak_memory_stats(device_id)
preflight_started = time.perf_counter()
try:
    with torch.inference_mode():
        preflight_model_inputs = preflight_pixel_values.to(
            device='cuda:0', dtype=BACKBONE_COMPUTE_DTYPE, non_blocking=True
        )
        preflight_pooled = feature_model(preflight_model_inputs)
except RuntimeError as exc:
    if 'out of memory' not in str(exc).lower():
        raise RuntimeError('Preflight H+ FP32 DataParallel gagal; tidak ada fallback otomatis.') from exc
    raise RuntimeError(
        'CUDA OOM pada preflight H+ FP32 384px, batch 2, T4 x2. Mulai sesi T4 x2 yang '
        'bersih dan ulangi konfigurasi yang sama. Jika OOM berulang, tandai konfigurasi '
        'tidak feasible; jangan ganti dtype, model, resolusi, GPU, atau batch secara otomatis.'
    ) from exc
for device_id in DEVICE_IDS:
    torch.cuda.synchronize(device_id)
preflight_elapsed_seconds = time.perf_counter() - preflight_started

expected_output_shape = (FEATURE_BATCH_SIZE, 2, HIDDEN_SIZE)
if tuple(preflight_pooled.shape) != expected_output_shape:
    raise RuntimeError(
        f'Output pooled preflight salah: {tuple(preflight_pooled.shape)}; '
        f'expected={expected_output_shape}.'
    )
if preflight_pooled.dtype != BACKBONE_COMPUTE_DTYPE:
    raise RuntimeError(f'Output pooled preflight wajib FP32, got {preflight_pooled.dtype}.')
if not torch.isfinite(preflight_pooled).all().item():
    raise RuntimeError('Output pooled preflight mengandung NaN/Inf.')
preflight_concat = torch.cat((preflight_pooled[:, 0], preflight_pooled[:, 1]), dim=1)
if tuple(preflight_concat.shape) != (FEATURE_BATCH_SIZE, PROBE_INPUT_DIM):
    raise RuntimeError(
        f'Dimensi concat preflight salah: {tuple(preflight_concat.shape)}; '
        f'expected={(FEATURE_BATCH_SIZE, PROBE_INPUT_DIM)}.'
    )
if PROBE_INPUT_DIM != 2 * HIDDEN_SIZE or not torch.isfinite(preflight_concat).all().item():
    raise RuntimeError('Kontrak concat 2*hidden_size atau finite preflight gagal.')
if any(parameter.requires_grad for parameter in feature_model.parameters()):
    raise RuntimeError('Backbone berubah menjadi trainable setelah preflight.')
if {parameter.dtype for parameter in feature_model.parameters()} != {BACKBONE_COMPUTE_DTYPE}:
    raise RuntimeError('Parameter backbone berubah dari FP32 setelah preflight.')
post_preflight_float_buffer_dtypes = {
    buffer.dtype for buffer in feature_model.buffers() if buffer.is_floating_point()
}
if not post_preflight_float_buffer_dtypes.issubset({BACKBONE_COMPUTE_DTYPE}):
    raise RuntimeError(
        f'Floating buffer berubah dari FP32 setelah preflight: {post_preflight_float_buffer_dtypes}.'
    )

preflight_gpu_memory = []
for device_id in DEVICE_IDS:
    peak_allocated_bytes = torch.cuda.max_memory_allocated(device_id)
    peak_reserved_bytes = torch.cuda.max_memory_reserved(device_id)
    if peak_allocated_bytes <= preflight_allocated_before[device_id] or peak_reserved_bytes <= 0:
        raise RuntimeError(
            f'Preflight tidak mengobservasi kenaikan alokasi CUDA pada cuda:{device_id}; '
            'asumsi split satu contoh per T4 tidak terpenuhi.'
        )
    preflight_gpu_memory.append({
        'device_id': device_id,
        'name': GPU_RUNTIME_INFO[device_id]['name'],
        'allocated_before_gib': preflight_allocated_before[device_id] / 2**30,
        'peak_allocated_gib': peak_allocated_bytes / 2**30,
        'peak_reserved_gib': peak_reserved_bytes / 2**30,
        'peak_allocated_delta_gib': (
            peak_allocated_bytes - preflight_allocated_before[device_id]
        ) / 2**30,
    })

preflight_contract = {
    'schema_version': PREFLIGHT_SCHEMA_VERSION,
    'revision_marker': V6_REVISION_MARKER,
    'model_id': MODEL_ID,
    'model_revision': RESOLVED_REVISION,
    'processor_signature': PREPROCESS_SIGNATURE,
    'synthetic_source_images': preflight_source_specs,
    'input_shape': list(expected_input_shape),
    'output_shape': list(expected_output_shape),
    'concat_shape': [FEATURE_BATCH_SIZE, PROBE_INPUT_DIM],
    'dtype': BACKBONE_COMPUTE_DTYPE_NAME,
    'hidden_size': HIDDEN_SIZE,
    'num_register_tokens': NUM_REGISTER_TOKENS,
    'patch_token_count': EXPECTED_PATCH_TOKENS,
    'sequence_token_count': EXPECTED_SEQUENCE_TOKENS,
    'transform_max_abs_tolerance': preflight_transform_max_abs_tolerance,
    'transform_mean_abs_tolerance': preflight_transform_mean_abs_tolerance,
    'feature_batch_size': FEATURE_BATCH_SIZE,
    'data_parallel_device_ids': DEVICE_IDS,
}
PREFLIGHT_CONTRACT_SIGNATURE = sha256_json(preflight_contract)
preflight_report = {
    'schema_version': PREFLIGHT_SCHEMA_VERSION,
    'revision_marker': V6_REVISION_MARKER,
    'contract_signature': PREFLIGHT_CONTRACT_SIGNATURE,
    'pass': True,
    'scope': 'minimal_in_memory_dual_t4_forward_only_not_full_dataset_feasibility',
    'model_id': MODEL_ID,
    'model_revision': RESOLVED_REVISION,
    'processor_class': PREPROCESS_CONFIG['processor_class'],
    'processor_signature': PREPROCESS_SIGNATURE,
    'synthetic_source_images': preflight_source_specs,
    'official_processor_matches_extraction_transform': True,
    'transform_max_abs_difference': preflight_transform_max_abs_difference,
    'transform_mean_abs_difference': preflight_transform_mean_abs_difference,
    'input_shape': list(preflight_pixel_values.shape),
    'output_shape': list(preflight_pooled.shape),
    'concat_shape': list(preflight_concat.shape),
    'dtype': BACKBONE_COMPUTE_DTYPE_NAME,
    'hidden_size': HIDDEN_SIZE,
    'num_register_tokens': NUM_REGISTER_TOKENS,
    'patch_token_count': EXPECTED_PATCH_TOKENS,
    'sequence_token_count': EXPECTED_SEQUENCE_TOKENS,
    'finite': True,
    'frozen': True,
    'feature_batch_size': FEATURE_BATCH_SIZE,
    'data_parallel_device_ids': DEVICE_IDS,
    'per_gpu_peak_memory': preflight_gpu_memory,
    'elapsed_seconds': preflight_elapsed_seconds,
}
write_json_atomic(preflight_report, PREFLIGHT_REPORT_PATH)
PREFLIGHT_PASSED = True
print('Preflight passed:', preflight_report)

del official_preflight_batch, preflight_pixel_values, manual_preflight_pixel_values
del preflight_abs_difference, preflight_model_inputs, preflight_pooled, preflight_concat
del preflight_images, x_axis, y_axis, x_grid, y_grid, channels, pattern, image
gc.collect()
torch.cuda.empty_cache()
for device_id in DEVICE_IDS:
    torch.cuda.synchronize(device_id)
    torch.cuda.reset_peak_memory_stats(device_id)


## 5. Index raw train, assignment fold, audit dimensi, dan safe-copy setelah preflight

Enumerasi tree train mentah, assignment fold, full metadata-image audit, dan pembuatan safe-copy baru hanya dimulai setelah preflight runtime menghasilkan `PREFLIGHT_PASSED=True`. File resmi tidak pernah ditimpa; salinan besar tetap berada di artifact directory v6 dengan signature preprocessing yang tercatat.

In [ ]:
if PREFLIGHT_PASSED is not True or not PREFLIGHT_REPORT_PATH.is_file():
    raise RuntimeError('Full image preprocessing diblokir karena preflight belum sukses.')

TRAIN_DIR = resolve_train_dir()
train_df = index_raw_train(TRAIN_DIR)
train_df['fold'] = -1
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, valid_indices) in enumerate(splitter.split(train_df, train_df['label'])):
    train_df.loc[valid_indices, 'fold'] = fold

if not train_df['fold'].between(0, N_FOLDS - 1).all():
    raise RuntimeError('Assignment fold tidak lengkap atau berada di luar 0..4.')
fold_class_counts = train_df.groupby(['fold', 'label']).size().unstack(fill_value=0)
if fold_class_counts.shape != (N_FOLDS, NUM_CLASSES) or not (fold_class_counts > 0).all().all():
    raise RuntimeError('Setiap fold wajib memiliki semua tiga kelas.')
if set(train_df.loc[train_df['fold'] != VALIDATION_FOLD, 'fold']) != {1, 2, 3, 4}:
    raise RuntimeError('Partition train v6 wajib tepat folds 1–4.')

fold_columns = ['image_id', 'relative_path', 'class_name', 'label', 'fold']
FOLD_ASSIGNMENT_SHA256 = sha256_json(
    train_df[['image_id', 'label', 'fold']].astype(str).values.tolist()
)
RAW_DATA_SHA256 = sha256_json(
    train_df[['image_id', 'label']].astype(str).values.tolist()
)
write_csv_atomic(train_df[fold_columns], ARTIFACT_DIR / 'fold_assignments.csv')
data_and_fold_signature = {
    'schema_version': 1,
    'source_policy': 'official_raw_train_labels_only',
    'class_directories': CLASS_DIR_TO_LABEL,
    'class_names': list(CLASS_NAMES),
    'raw_class_counts': {str(key): value for key, value in EXPECTED_RAW_COUNTS.items()},
    'row_count': len(train_df),
    'ordered_image_ids_sha256': sha256_json(train_df['image_id'].tolist()),
    'raw_data_sha256': RAW_DATA_SHA256,
    'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256,
    'fold_definition': {
        'implementation': 'sklearn.model_selection.StratifiedKFold',
        'n_splits': N_FOLDS,
        'shuffle': True,
        'random_state': SEED,
        'row_order': 'image_id ascending',
        'validation_fold': VALIDATION_FOLD,
        'training_folds': [1, 2, 3, 4],
    },
    'fold_class_counts': {
        str(fold): {str(label): int(fold_class_counts.loc[fold, label]) for label in range(NUM_CLASSES)}
        for fold in range(N_FOLDS)
    },
    'duplicate_policy': 'official rows retained; no duplicate removal',
    'relabeling': False,
}
write_json_atomic(data_and_fold_signature, ARTIFACT_DIR / 'data_and_fold_signature.json')
print('Resolved official raw train:', TRAIN_DIR)
print('Raw counts:', train_df['label'].value_counts().sort_index().to_dict())
print('Fold/class audit:\n', fold_class_counts)

run_df = prepare_safe_image_paths(train_df)
if len(run_df) != EXPECTED_TRAIN_ROWS:
    raise RuntimeError('Safe preprocessing mengubah jumlah row train.')


## 6. Cache fitur original-view yang strict dan dapat dilanjutkan

Setiap shard menyimpan pasangan `[CLS, mean_true_patch]`, bukan concat yang tidak dapat diaudit. Output backbone divalidasi saat masih FP32, dicek terhadap batas FP16, baru kemudian dicast dan divalidasi lagi. Tidak ada clamp, penggantian NaN/Inf, retry otomatis, atau reuse cache v5.

Identitas cache hanya memuat faktor yang menentukan frozen features—data dan urutannya, model/revision, preprocessing, image size/view, precision, serta kontrak token/pooling. Budget epoch dan recipe head sengaja tidak menjadi identitas cache. Shard yang lengkap dan identik provenance-nya boleh dipakai ulang; shard inkompatibel ditolak dan dihitung ulang secara atomik.

In [ ]:
if PREFLIGHT_PASSED is not True:
    raise RuntimeError('Feature-cache extraction diblokir karena preflight belum sukses.')

def host_rss_gib():
    try:
        for line in Path('/proc/self/status').read_text(encoding='utf-8').splitlines():
            if line.startswith('VmRSS:'):
                return int(line.split()[1]) / 1024**2
    except OSError:
        return None
    return None

def gpu_memory_snapshot():
    rows = []
    for device_id in DEVICE_IDS:
        torch.cuda.synchronize(device_id)
        rows.append({
            'device_id': device_id,
            'allocated_gib': torch.cuda.memory_allocated(device_id) / 2**30,
            'reserved_gib': torch.cuda.memory_reserved(device_id) / 2**30,
            'peak_allocated_gib': torch.cuda.max_memory_allocated(device_id) / 2**30,
            'peak_reserved_gib': torch.cuda.max_memory_reserved(device_id) / 2**30,
        })
    return rows

def shard_ranges(row_count):
    return [
        (shard_index, start, min(start + FEATURE_SHARD_SIZE, row_count))
        for shard_index, start in enumerate(range(0, row_count, FEATURE_SHARD_SIZE))
    ]

CACHE_NAMESPACE_CONFIG = {
    'schema_version': CACHE_SCHEMA_VERSION,
    'identity_scope': 'frozen_backbone_features_only_no_probe_hyperparameters',
    'split': 'train',
    'view': FEATURE_VIEW,
    'model_id': MODEL_ID,
    'model_revision': RESOLVED_REVISION,
    'transformers_version': transformers.__version__,
    'torch_version': torch.__version__,
    'backbone_compute_dtype': BACKBONE_COMPUTE_DTYPE_NAME,
    'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
    'attention_implementation': ATTENTION_IMPLEMENTATION,
    'image_size': IMAGE_SIZE,
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'feature_representation': 'stack(cls, mean_true_patch_tokens)',
    'hidden_size': HIDDEN_SIZE,
    'num_register_tokens': NUM_REGISTER_TOKENS,
    'patch_token_count': EXPECTED_PATCH_TOKENS,
    'sequence_token_count': EXPECTED_SEQUENCE_TOKENS,
    'feature_shard_size': FEATURE_SHARD_SIZE,
    'ordered_image_ids_sha256': sha256_json(run_df['image_id'].astype(str).tolist()),
    'labels_sha256': sha256_json(run_df['label'].astype(int).tolist()),
    'raw_data_sha256': RAW_DATA_SHA256,
}
CACHE_NAMESPACE = SHARDS_DIR / f'train_orig_{sha256_json(CACHE_NAMESPACE_CONFIG)[:16]}'
CACHE_NAMESPACE.mkdir(parents=True, exist_ok=True)

def expected_shard_metadata(shard_index, start, end):
    shard_frame = run_df.iloc[start:end]
    return {
        'schema_version': CACHE_SCHEMA_VERSION,
        'identity_scope': 'frozen_backbone_features_only_no_probe_hyperparameters',
        'split': 'train',
        'view': FEATURE_VIEW,
        'shard_index': int(shard_index),
        'start_index': int(start),
        'end_index': int(end),
        'row_range': [int(start), int(end)],
        'row_count': int(end - start),
        'ordered_image_ids': shard_frame['image_id'].astype(str).tolist(),
        'labels': shard_frame['label'].astype(int).tolist(),
        'model_id': MODEL_ID,
        'model_revision': RESOLVED_REVISION,
        'transformers_version': transformers.__version__,
        'torch_version': torch.__version__,
        'backbone_compute_dtype': BACKBONE_COMPUTE_DTYPE_NAME,
        'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
        'attention_implementation': ATTENTION_IMPLEMENTATION,
        'image_size': IMAGE_SIZE,
        'processor_class': type(processor).__name__,
        'processor_config': PREPROCESS_CONFIG,
        'preprocess_signature': PREPROCESS_SIGNATURE,
        'feature_representation': 'stack(cls, mean_true_patch_tokens)',
        'feature_shape': [int(end - start), 2, HIDDEN_SIZE],
        'hidden_size': HIDDEN_SIZE,
        'num_register_tokens': NUM_REGISTER_TOKENS,
        'patch_token_count': EXPECTED_PATCH_TOKENS,
        'sequence_token_count': EXPECTED_SEQUENCE_TOKENS,
        'raw_class_counts': {str(key): value for key, value in EXPECTED_RAW_COUNTS.items()},
        'raw_data_sha256': RAW_DATA_SHA256,
    }

def shard_path(shard_index):
    return CACHE_NAMESPACE / f'train_orig_shard_{shard_index:03d}.pt'

def validate_feature_shard(payload, expected_metadata):
    if not isinstance(payload, dict) or not isinstance(payload.get('metadata'), dict):
        raise RuntimeError('Payload/metadata shard hilang atau tidak valid.')
    if payload['metadata'] != expected_metadata:
        raise RuntimeError('Metadata/provenance shard tidak cocok persis.')
    features = payload.get('features')
    expected_shape = tuple(expected_metadata['feature_shape'])
    if not isinstance(features, torch.Tensor) or tuple(features.shape) != expected_shape:
        raise RuntimeError(f'Shape shard salah; got={getattr(features, "shape", None)}, expected={expected_shape}.')
    if features.dtype != FEATURE_STORAGE_DTYPE or not torch.isfinite(features).all().item():
        raise RuntimeError('Fitur shard wajib finite FP16.')
    labels = payload.get('labels')
    if (
        not isinstance(labels, torch.Tensor)
        or labels.dtype != torch.long
        or labels.tolist() != expected_metadata['labels']
    ):
        raise RuntimeError('Label atau urutan label shard tidak cocok.')
    return payload

def finite_min_max(tensor):
    finite_values = tensor[torch.isfinite(tensor)]
    if not finite_values.numel():
        return None
    return [float(finite_values.min().item()), float(finite_values.max().item())]

def affected_examples(shard_frame, batch_rows, affected_batch_rows, limit=5):
    examples = []
    for batch_row in affected_batch_rows[:limit]:
        shard_row = batch_rows[batch_row]
        record = shard_frame.iloc[shard_row]
        examples.append({
            'batch_row': int(batch_row),
            'shard_row': int(shard_row),
            'image_id': str(record['image_id']),
            'path': str(record['path']),
        })
    return examples

def validate_fp32_pooled_batch(pooled, model_inputs, batch_rows, shard_frame, shard_index):
    expected_shape = (len(batch_rows), 2, HIDDEN_SIZE)
    context = f'split=train, view=orig, shard={shard_index:03d}, compute_dtype=float32'
    if tuple(pooled.shape) != expected_shape or pooled.dtype != BACKBONE_COMPUTE_DTYPE:
        raise RuntimeError(
            f'Pooled contract sebelum cast salah: shape={tuple(pooled.shape)}, dtype={pooled.dtype}, '
            f'expected={expected_shape}/float32; {context}.'
        )
    finite_mask = torch.isfinite(pooled)
    if finite_mask.all().item():
        return
    affected_batch_rows = torch.nonzero(
        (~finite_mask).flatten(1).any(dim=1), as_tuple=False
    ).flatten().tolist()
    raise RuntimeError(
        'Non-finite FP32 pooled feature sebelum storage cast; '
        f'{context}; total_nonfinite={int((~finite_mask).sum().item())}; '
        f'nan={int(torch.isnan(pooled).sum().item())}; inf={int(torch.isinf(pooled).sum().item())}; '
        f'affected={json.dumps(affected_examples(shard_frame, batch_rows, affected_batch_rows), ensure_ascii=False)}; '
        f'input_finite_min_max={finite_min_max(model_inputs)}; '
        f'pooled_finite_min_max={finite_min_max(pooled)}. Tidak ada clamp/imputasi.'
    )

def extract_feature_shard(shard_frame, metadata, output_path):
    dataset = OriginalViewTrainDataset(shard_frame)
    loader = DataLoader(
        dataset,
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=FEATURE_WORKERS,
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2,
        drop_last=False,
    )
    feature_parts = []
    observed_labels = []
    observed_indices = []
    started = time.perf_counter()
    with torch.inference_mode():
        for images, labels, indices in tqdm(
            loader, desc=f'train/orig shard {metadata["shard_index"]:03d}', leave=False
        ):
            batch_rows = [int(index) for index in indices.tolist()]
            if images.dtype != BACKBONE_COMPUTE_DTYPE:
                raise RuntimeError(f'DataLoader image wajib FP32, got {images.dtype}.')
            input_finite_mask = torch.isfinite(images)
            if not input_finite_mask.all().item():
                affected_batch_rows = torch.nonzero(
                    (~input_finite_mask).flatten(1).any(dim=1), as_tuple=False
                ).flatten().tolist()
                raise RuntimeError(
                    'Non-finite FP32 input sebelum backbone; '
                    f'shard={metadata["shard_index"]:03d}; '
                    f'total_nonfinite={int((~input_finite_mask).sum().item())}; '
                    f'nan={int(torch.isnan(images).sum().item())}; inf={int(torch.isinf(images).sum().item())}; '
                    f'affected={json.dumps(affected_examples(shard_frame, batch_rows, affected_batch_rows), ensure_ascii=False)}; '
                    f'finite_min_max={finite_min_max(images)}.'
                )
            model_inputs = images.to(
                device='cuda:0', dtype=BACKBONE_COMPUTE_DTYPE, non_blocking=True
            )
            try:
                pooled = feature_model(model_inputs)
            except RuntimeError as exc:
                if 'out of memory' not in str(exc).lower():
                    raise
                raise RuntimeError(
                    'CUDA OOM pada H+ FP32 384px dengan total batch=2 (satu gambar/T4). '
                    'Mulai Kaggle T4 x2 yang bersih dan rerun dengan FEATURE_BATCH_SIZE tetap 2. '
                    'Jika OOM berulang, tandai konfigurasi ini tidak feasible; jangan otomatis mengganti '
                    'dtype/model/resolusi/GPU. Batch=1 hanya boleh dicoba manual sebagai diagnosis terpisah '
                    'dan bukan bukti throughput dual-GPU. Tidak ada retry otomatis.'
                ) from exc
            validate_fp32_pooled_batch(
                pooled, model_inputs, batch_rows, shard_frame, metadata['shard_index']
            )
            max_abs = float(pooled.abs().max().item())
            fp16_limit = float(torch.finfo(torch.float16).max)
            if max_abs > fp16_limit:
                raise RuntimeError(
                    f'FP32 pooled feature melampaui range FP16: max_abs={max_abs}, '
                    f'limit={fp16_limit}, shard={metadata["shard_index"]:03d}. '
                    'Jangan cast/clamp; ubah storage dtype hanya lewat eksperimen baru yang ditinjau.'
                )
            stored = pooled.detach().to(device='cpu', dtype=FEATURE_STORAGE_DTYPE)
            stored_finite_mask = torch.isfinite(stored)
            if stored.dtype != FEATURE_STORAGE_DTYPE or not stored_finite_mask.all().item():
                affected_batch_rows = torch.nonzero(
                    (~stored_finite_mask).flatten(1).any(dim=1), as_tuple=False
                ).flatten().tolist()
                raise RuntimeError(
                    'Cast storage FP16 gagal validasi; '
                    f'nonfinite={int((~stored_finite_mask).sum().item())}, '
                    f'affected_batch_rows={affected_batch_rows}, shard={metadata["shard_index"]:03d}. '
                    'Tidak ada nilai yang disimpan.'
                )
            feature_parts.append(stored)
            observed_labels.extend(labels.tolist())
            observed_indices.extend(indices.tolist())
    if observed_indices != list(range(len(shard_frame))):
        raise RuntimeError(f'Urutan DataLoader berubah untuk {output_path.name}.')
    features = torch.cat(feature_parts, dim=0)
    labels_tensor = torch.tensor(observed_labels, dtype=torch.long)
    payload = {'metadata': metadata, 'features': features, 'labels': labels_tensor}
    validate_feature_shard(payload, metadata)
    atomic_torch_save(payload, output_path)
    return time.perf_counter() - started

feature_runtime = {
    'completed': False,
    'rows': len(run_df),
    'extracted_rows': 0,
    'reused_rows': 0,
    'rejected_shards': 0,
    'extraction_seconds': 0.0,
}
cache_started = time.perf_counter()
shard_paths = []
for shard_index, start, end in shard_ranges(len(run_df)):
    metadata = expected_shard_metadata(shard_index, start, end)
    path = shard_path(shard_index)
    reused = False
    if path.is_file():
        try:
            validate_feature_shard(
                torch.load(path, map_location='cpu', weights_only=False), metadata
            )
            reused = True
            feature_runtime['reused_rows'] += end - start
        except Exception as exc:
            print(f'Shard ditolak dan dihitung ulang: {path.name} ({exc})')
            path.unlink()
            feature_runtime['rejected_shards'] += 1
    if not reused:
        shard_frame = run_df.iloc[start:end].reset_index(drop=True)
        elapsed = extract_feature_shard(shard_frame, metadata, path)
        feature_runtime['extraction_seconds'] += elapsed
        feature_runtime['extracted_rows'] += end - start
    shard_paths.append(str(path))
    memory = gpu_memory_snapshot()
    rss = host_rss_gib()
    throughput = (
        feature_runtime['extracted_rows'] / feature_runtime['extraction_seconds']
        if feature_runtime['extraction_seconds'] > 0 else None
    )
    memory_text = ', '.join(
        f"gpu{row['device_id']} alloc={row['allocated_gib']:.2f}G "
        f"reserved={row['reserved_gib']:.2f}G peak={row['peak_allocated_gib']:.2f}G"
        for row in memory
    )
    status_parts = [
        f'shard {shard_index:03d} {"reused" if reused else "written"}',
        f'elapsed={time.perf_counter() - cache_started:.1f}s',
        f'new_throughput={throughput:.2f} img/s' if throughput is not None else 'new_throughput=n/a',
        f'host_rss={rss:.2f} GiB' if rss is not None else 'host_rss=n/a',
        memory_text,
    ]
    print('; '.join(status_parts))

feature_parts = []
label_parts = []
reconstructed_ids = []
for shard_index, start, end in shard_ranges(len(run_df)):
    metadata = expected_shard_metadata(shard_index, start, end)
    path = shard_path(shard_index)
    if not path.is_file():
        raise RuntimeError(f'Shard wajib hilang setelah extraction/resume: {path}.')
    payload = validate_feature_shard(
        torch.load(path, map_location='cpu', weights_only=False), metadata
    )
    feature_parts.append(payload['features'])
    label_parts.append(payload['labels'])
    reconstructed_ids.extend(metadata['ordered_image_ids'])
cached_feature_pairs = torch.cat(feature_parts, dim=0)
cached_labels = torch.cat(label_parts, dim=0)
expected_ids = run_df['image_id'].astype(str).tolist()
expected_labels = run_df['label'].astype(int).tolist()
if reconstructed_ids != expected_ids or cached_labels.tolist() != expected_labels:
    raise RuntimeError('Rekonstruksi cache mengubah urutan image_id atau labels.')
if tuple(cached_feature_pairs.shape) != (len(run_df), 2, HIDDEN_SIZE):
    raise RuntimeError(f'Shape cache gabungan salah: {tuple(cached_feature_pairs.shape)}.')
if cached_feature_pairs.dtype != FEATURE_STORAGE_DTYPE or not torch.isfinite(cached_feature_pairs).all().item():
    raise RuntimeError('Cache gabungan wajib finite FP16.')

feature_runtime['cache_handling_seconds'] = time.perf_counter() - cache_started
feature_runtime['new_extraction_throughput_images_per_second'] = (
    feature_runtime['extracted_rows'] / feature_runtime['extraction_seconds']
    if feature_runtime['extraction_seconds'] > 0 else None
)
feature_runtime['host_rss_gib_at_completion'] = host_rss_gib()
feature_runtime['gpu_memory_at_completion'] = gpu_memory_snapshot()
feature_runtime['completed'] = True
feature_cache_manifest = {
    **CACHE_NAMESPACE_CONFIG,
    'run_revision_marker': V6_REVISION_MARKER,
    'namespace': str(CACHE_NAMESPACE),
    'row_count': len(run_df),
    'shard_count': len(shard_paths),
    'shard_paths': shard_paths,
    'cached_feature_shape': list(cached_feature_pairs.shape),
    'feature_runtime': feature_runtime,
}
write_json_atomic(feature_cache_manifest, ARTIFACT_DIR / 'feature_cache_manifest.json')
print('Feature cache complete and aligned:', tuple(cached_feature_pairs.shape))

# Backbone H+ dilepas sebelum probe kecil dilatih.
del feature_model, pooler, backbone, processor, feature_parts, label_parts
gc.collect()
torch.cuda.empty_cache()
for device_id in DEVICE_IDS:
    torch.cuda.synchronize(device_id)
print('Frozen H+ released; GPU cache cleared before probe training.')


## 7. Probe terkunci: fold 0, seed 42, original view

Fitur FP16 cache dikonversi kembali ke FP32 dan di-concat menjadi `2 × hidden_size` secara dinamis. Train memakai folds 1–4; validation hanya fold 0. Bobot kelas dihitung **hanya** dari label partition train dengan normalisasi inverse-frequency yang sama dengan v5.

Head tetap: `LayerNorm → Linear(512) → GELU → Dropout(0,20) → Linear(3)`, AdamW LR 8e-4, WD 1e-4, cosine annealing, maksimal 15 epoch, patience 5, clip 1,0. Tepat satu run memakai seed 42; tidak ada loop, averaging, ensemble, atau confirmation run seed lain. Best epoch dipilih oleh macro F1 validation; log loss menjadi tie-break deterministik.

In [ ]:
concat_features = torch.cat(
    (cached_feature_pairs[:, 0], cached_feature_pairs[:, 1]), dim=1
).to(dtype=torch.float32).contiguous()
if tuple(concat_features.shape) != (len(run_df), PROBE_INPUT_DIM):
    raise RuntimeError(
        f'Dimensi concat salah: {tuple(concat_features.shape)}; '
        f'expected={(len(run_df), PROBE_INPUT_DIM)}.'
    )
if not torch.isfinite(concat_features).all().item():
    raise RuntimeError('Concat probe features mengandung NaN/Inf setelah konversi FP32.')
del cached_feature_pairs, cached_labels
gc.collect()

fold_values = run_df['fold'].to_numpy(dtype=np.int64)
labels = run_df['label'].to_numpy(dtype=np.int64)
train_indices = np.flatnonzero(fold_values != VALIDATION_FOLD)
valid_indices = np.flatnonzero(fold_values == VALIDATION_FOLD)
if not len(train_indices) or not len(valid_indices):
    raise RuntimeError('Partition train/validation fold 0 kosong.')
if set(fold_values[train_indices]) != {1, 2, 3, 4} or set(fold_values[valid_indices]) != {0}:
    raise RuntimeError('Membership probe bukan train folds 1–4 versus validation fold 0.')
if set(labels[valid_indices]) != set(range(NUM_CLASSES)):
    raise RuntimeError('Validation fold 0 tidak memiliki semua tiga kelas.')

train_features = concat_features[train_indices]
train_labels = torch.from_numpy(labels[train_indices]).long()
valid_features = concat_features[valid_indices]
valid_labels = labels[valid_indices]
train_class_counts = np.bincount(labels[train_indices], minlength=NUM_CLASSES)
if (train_class_counts <= 0).any():
    raise RuntimeError(f'Class count train invalid: {train_class_counts.tolist()}.')
class_weights = torch.tensor(
    train_class_counts.sum() / (NUM_CLASSES * train_class_counts),
    dtype=torch.float32,
    device='cuda:0',
)

class Probe(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, features):
        return self.network(features)

def softmax_numpy(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    probabilities = exponentials / exponentials.sum(axis=1, keepdims=True)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('Probabilitas probe mengandung NaN/Inf.')
    return probabilities

def predict_probe(model, features):
    loader = DataLoader(
        TensorDataset(features, torch.arange(len(features), dtype=torch.long)),
        batch_size=PREDICT_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    parts = []
    observed_indices = []
    model.eval()
    with torch.inference_mode():
        for batch, batch_indices in loader:
            logits = model(batch.to(device='cuda:0', dtype=torch.float32, non_blocking=True))
            parts.append(logits.cpu())
            observed_indices.extend(batch_indices.tolist())
    if observed_indices != list(range(len(features))):
        raise RuntimeError('Urutan prediction DataLoader berubah; menolak prediksi misaligned.')
    result = torch.cat(parts).numpy().astype(np.float32)
    if result.shape != (len(features), NUM_CLASSES) or not np.isfinite(result).all():
        raise RuntimeError(f'Output prediction tidak lengkap/finite: shape={result.shape}.')
    return result

def checkpoint_metadata():
    return {
        'schema_version': 3,
        'revision_marker': V6_REVISION_MARKER,
        'experiment_name': EXPERIMENT_NAME,
        'model_id': MODEL_ID,
        'model_revision': RESOLVED_REVISION,
        'preprocess_signature': PREPROCESS_SIGNATURE,
        'preflight_contract_signature': PREFLIGHT_CONTRACT_SIGNATURE,
        'torch_version': torch.__version__,
        'transformers_version': transformers.__version__,
        'backbone_compute_dtype': BACKBONE_COMPUTE_DTYPE_NAME,
        'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
        'attention_implementation': ATTENTION_IMPLEMENTATION,
        'hidden_size': HIDDEN_SIZE,
        'num_register_tokens': NUM_REGISTER_TOKENS,
        'patch_token_count': EXPECTED_PATCH_TOKENS,
        'raw_data_sha256': RAW_DATA_SHA256,
        'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256,
        'recipe_name': PROBE_RECIPE_NAME,
        'pooling': POOLING_REPRESENTATION,
        'input_dim': PROBE_INPUT_DIM,
        'validation_fold': VALIDATION_FOLD,
        'training_folds': [1, 2, 3, 4],
        'seed': SEED,
        'max_epochs': MAX_EPOCHS,
        'epoch_budget_source': 'matched_to_completed_v5_full_probe_FULL_EPOCHS_15',
        'patience': EARLY_STOPPING_PATIENCE,
        'learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': HEAD_WEIGHT_DECAY,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
        'feature_view': FEATURE_VIEW,
        'comparison_role': 'v6_compute_efficient_screen_not_protocol_matched_to_v5_fullrun',
        'v5_reference_protocol': {
            'probe_training_views': ['orig', 'hflip'],
            'head_seeds': [42, 314],
            'max_epochs': 15,
        },
        'v6_screen_protocol': {
            'probe_training_views': ['orig'],
            'head_seeds': [42],
        },
        'class_weights': class_weights.detach().cpu().tolist(),
    }

CHECKPOINT_PATH = ARTIFACT_DIR / 'head_fold0_seed42.pt'
expected_checkpoint_metadata = checkpoint_metadata()
existing_head_paths = set(ARTIFACT_DIR.rglob('head_*.pt'))
unexpected_head_paths = existing_head_paths - {CHECKPOINT_PATH}
if unexpected_head_paths:
    raise RuntimeError(f'Checkpoint v6 tak terduga/stale: {sorted(map(str, unexpected_head_paths))}.')
if CHECKPOINT_PATH.is_file():
    prior_checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    if prior_checkpoint.get('metadata') != expected_checkpoint_metadata:
        raise RuntimeError(
            'Checkpoint fold0 seed42 yang ada stale atau provenance-nya tidak cocok dengan kontrak '
            'final epoch-15. Hapus atau pindahkan checkpoint stale, lalu rerun; file ini '
            'tidak akan dipakai ulang secara diam-diam.'
        )
    print('Checkpoint kompatibel dari run terdahulu akan diganti atomik oleh retraining deterministik.')

seed_everything(SEED)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    TensorDataset(train_features, train_labels),
    batch_size=HEAD_BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)
probe = Probe(PROBE_INPUT_DIM).to('cuda:0')
PROBE_PARAMETER_COUNT = sum(parameter.numel() for parameter in probe.parameters())
PROBE_TRAINABLE_PARAMETER_COUNT = sum(
    parameter.numel() for parameter in probe.parameters() if parameter.requires_grad
)
if PROBE_PARAMETER_COUNT != PROBE_TRAINABLE_PARAMETER_COUNT:
    raise RuntimeError('Seluruh parameter probe seharusnya trainable.')
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    probe.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

best_state = None
best_epoch = None
best_macro_f1 = -1.0
best_log_loss = float('inf')
stale_epochs = 0
history = []
probe_started = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started = time.perf_counter()
    probe.train()
    loss_sum = 0.0
    seen = 0
    learning_rate_used = optimizer.param_groups[0]['lr']
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to('cuda:0', dtype=torch.float32, non_blocking=True)
        batch_labels = batch_labels.to('cuda:0', non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = probe(batch_features)
        loss = criterion(logits, batch_labels)
        if not torch.isfinite(loss).item():
            raise RuntimeError(f'Probe loss non-finite pada epoch {epoch}.')
        loss.backward()
        nn.utils.clip_grad_norm_(probe.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
        optimizer.step()
        loss_sum += loss.item() * len(batch_labels)
        seen += len(batch_labels)
    scheduler.step()

    valid_logits = predict_probe(probe, valid_features)
    valid_probabilities = softmax_numpy(valid_logits)
    valid_predictions = valid_probabilities.argmax(axis=1)
    valid_macro_f1 = f1_score(
        valid_labels, valid_predictions, labels=list(range(NUM_CLASSES)), average='macro'
    )
    valid_log_loss = log_loss(
        valid_labels, valid_probabilities, labels=list(range(NUM_CLASSES))
    )
    epoch_seconds = time.perf_counter() - epoch_started
    history.append({
        'revision_marker': V6_REVISION_MARKER,
        'metric_role': 'v6_compute_efficient_fold0_screen',
        'validation_fold': VALIDATION_FOLD,
        'seed': SEED,
        'max_epochs_budget': MAX_EPOCHS,
        'epoch': epoch,
        'train_loss': loss_sum / seen,
        'valid_macro_f1': valid_macro_f1,
        'valid_log_loss': valid_log_loss,
        'learning_rate': learning_rate_used,
        'epoch_seconds': epoch_seconds,
    })
    print(
        f'epoch={epoch:02d} train_loss={loss_sum / seen:.6f} '
        f'val_f1={valid_macro_f1:.6f} val_logloss={valid_log_loss:.6f} '
        f'lr={learning_rate_used:.3e} time={epoch_seconds:.1f}s'
    )
    improved = (
        valid_macro_f1 > best_macro_f1 + 1e-8
        or (abs(valid_macro_f1 - best_macro_f1) <= 1e-8 and valid_log_loss < best_log_loss)
    )
    if improved:
        best_macro_f1 = float(valid_macro_f1)
        best_log_loss = float(valid_log_loss)
        best_epoch = epoch
        best_state = {
            name: value.detach().cpu().clone() for name, value in probe.state_dict().items()
        }
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(f'Early stopping pada epoch {epoch}; best epoch={best_epoch}.')
            break

if best_state is None or best_epoch is None:
    raise RuntimeError('Probe training tidak menghasilkan checkpoint valid.')
PROBE_TRAINING_SECONDS = time.perf_counter() - probe_started
history_df = pd.DataFrame(history)
write_csv_atomic(history_df, ARTIFACT_DIR / 'head_history_fold0_seed42.csv')
checkpoint_payload = {
    'metadata': expected_checkpoint_metadata,
    'best_epoch': best_epoch,
    'best_valid_macro_f1': best_macro_f1,
    'best_valid_log_loss': best_log_loss,
    'state_dict': best_state,
}
atomic_torch_save(checkpoint_payload, CHECKPOINT_PATH)
saved_checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
if saved_checkpoint.get('metadata') != expected_checkpoint_metadata:
    raise RuntimeError('Checkpoint atomik gagal validasi metadata setelah save.')
actual_head_paths = set(ARTIFACT_DIR.rglob('head_*.pt'))
if actual_head_paths != {CHECKPOINT_PATH}:
    raise RuntimeError(f'Membership checkpoint tidak eksak: {sorted(map(str, actual_head_paths))}.')

probe.load_state_dict(best_state, strict=True)
fold0_logits = predict_probe(probe, valid_features)
if fold0_logits.shape != (len(valid_indices), NUM_CLASSES):
    raise RuntimeError('Prediksi fold 0 tidak lengkap atau shape salah.')
del probe, train_loader, train_features, train_labels, valid_features, concat_features
gc.collect()
torch.cuda.empty_cache()


## 8. Evaluasi fold 0, diagnostik, dan project decision gate

Seluruh metrik runtime di bawah berasal dari v6 fold 0 original-view. Tabel referensi menampilkan 0,985022 dari v5 full run, tetapi protokol training-view dan seed-nya berbeda; karena itu selisih numerik bukan estimasi efek murni backbone. Notebook juga tidak mengarang referensi fold-0 per kelas yang tidak tersedia.

Project decision gate sudah dikunci sebelum hasil terlihat: `≥0,9880 = go`, `0,9865–<0,9880 = conditional`, `<0,9865 = stop`. Gate ini adalah aturan praktis untuk pekerjaan berikutnya, bukan bukti bahwa H+ secara kausal mengalahkan ViT-L. Kelayakan operasional dilaporkan dari runtime T4 ×2 yang benar-benar mencapai cell akhir.

In [ ]:
fold0_probabilities = softmax_numpy(fold0_logits)
if not np.allclose(fold0_probabilities.sum(axis=1), 1.0, atol=1e-6):
    raise RuntimeError('Probabilitas fold 0 tidak menjumlah ke satu.')
fold0_predictions = fold0_probabilities.argmax(axis=1).astype(np.int64)
if len(fold0_predictions) != len(valid_indices):
    raise RuntimeError('Jumlah prediksi fold 0 tidak sama dengan membership validation.')
fold0_macro_f1 = f1_score(
    valid_labels, fold0_predictions, labels=list(range(NUM_CLASSES)), average='macro'
)
fold0_accuracy = accuracy_score(valid_labels, fold0_predictions)
fold0_log_loss = log_loss(
    valid_labels, fold0_probabilities, labels=list(range(NUM_CLASSES))
)
report = classification_report(
    valid_labels,
    fold0_predictions,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
matrix = confusion_matrix(
    valid_labels, fold0_predictions, labels=list(range(NUM_CLASSES))
)
if matrix.shape != (NUM_CLASSES, NUM_CLASSES) or (matrix.sum(axis=1) == 0).any():
    raise RuntimeError('Confusion matrix tidak mencakup semua tiga kelas.')
normalized_matrix = np.divide(
    matrix,
    matrix.sum(axis=1, keepdims=True),
    out=np.zeros_like(matrix, dtype=np.float64),
    where=matrix.sum(axis=1, keepdims=True) != 0,
)

fold0_frame = run_df.iloc[valid_indices].copy().reset_index(drop=True)
if not (fold0_frame['fold'] == VALIDATION_FOLD).all():
    raise RuntimeError('Urutan/membership fold0_frame tidak cocok valid_indices.')
confidence = fold0_probabilities.max(axis=1)
sorted_probabilities = np.sort(fold0_probabilities, axis=1)
margin = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
prediction_table = fold0_frame[
    ['image_id', 'source_path', 'path', 'class_name', 'label', 'fold']
].copy()
prediction_table = prediction_table.rename(columns={'source_path': 'source_image_path', 'path': 'effective_image_path'})
prediction_table['prediction'] = fold0_predictions
prediction_table['predicted_class_name'] = [CLASS_NAMES[index] for index in fold0_predictions]
prediction_table['confidence'] = confidence
prediction_table['margin'] = margin
prediction_table['correct'] = fold0_predictions == valid_labels
prediction_table['metric_role'] = 'v6_fold0_original_view_compute_efficient_screen'
for class_id in range(NUM_CLASSES):
    prediction_table[f'prob_{class_id}'] = fold0_probabilities[:, class_id]
write_csv_atomic(prediction_table, ARTIFACT_DIR / 'fold0_predictions.csv')

if any(class_name not in report for class_name in CLASS_NAMES):
    raise RuntimeError('Classification report tidak memuat seluruh tiga kelas.')
report_frame = pd.DataFrame(report).T
report_frame.index.name = 'class_or_average'
write_csv_atomic(report_frame.reset_index(), ARTIFACT_DIR / 'classification_report.csv')
confusion_rows = []
for true_label in range(NUM_CLASSES):
    for predicted_label in range(NUM_CLASSES):
        confusion_rows.append({
            'true_label': true_label,
            'true_class': CLASS_NAMES[true_label],
            'predicted_label': predicted_label,
            'predicted_class': CLASS_NAMES[predicted_label],
            'count': int(matrix[true_label, predicted_label]),
            'row_normalized': float(normalized_matrix[true_label, predicted_label]),
        })
write_csv_atomic(pd.DataFrame(confusion_rows), ARTIFACT_DIR / 'confusion_matrix.csv')

high_confidence_errors = prediction_table.loc[~prediction_table['correct']].sort_values(
    ['confidence', 'margin', 'image_id'], ascending=[False, False, True]
).reset_index(drop=True)
write_csv_atomic(high_confidence_errors, ARTIFACT_DIR / 'high_confidence_errors.csv')

def distribution_summary(values):
    return {
        'mean': float(np.mean(values)),
        'std': float(np.std(values)),
        'min': float(np.min(values)),
        'p05': float(np.quantile(values, 0.05)),
        'p25': float(np.quantile(values, 0.25)),
        'median': float(np.quantile(values, 0.50)),
        'p75': float(np.quantile(values, 0.75)),
        'p95': float(np.quantile(values, 0.95)),
        'max': float(np.max(values)),
    }

prediction_distribution = {
    CLASS_NAMES[class_id]: int((fold0_predictions == class_id).sum())
    for class_id in range(NUM_CLASSES)
}
pairwise_confusions = {
    'recyclable_as_organic': int(matrix[0, 2]),
    'organic_as_recyclable': int(matrix[2, 0]),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
raw_image = axes[0, 0].imshow(matrix, cmap='Blues')
for row in range(NUM_CLASSES):
    for column in range(NUM_CLASSES):
        axes[0, 0].text(column, row, str(matrix[row, column]), ha='center', va='center')
axes[0, 0].set_title('Fold 0 raw confusion')
axes[0, 0].set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=20)
axes[0, 0].set_yticks(range(NUM_CLASSES), CLASS_NAMES)
axes[0, 0].set_xlabel('predicted')
axes[0, 0].set_ylabel('true')
fig.colorbar(raw_image, ax=axes[0, 0], fraction=0.046)

normalized_image = axes[0, 1].imshow(normalized_matrix, cmap='Blues', vmin=0, vmax=1)
for row in range(NUM_CLASSES):
    for column in range(NUM_CLASSES):
        axes[0, 1].text(
            column, row, f'{normalized_matrix[row, column]:.2%}', ha='center', va='center'
        )
axes[0, 1].set_title(f'Row-normalized — macro F1 {fold0_macro_f1:.6f}')
axes[0, 1].set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=20)
axes[0, 1].set_yticks(range(NUM_CLASSES), CLASS_NAMES)
axes[0, 1].set_xlabel('predicted')
axes[0, 1].set_ylabel('true')
fig.colorbar(normalized_image, ax=axes[0, 1], fraction=0.046)

axes[1, 0].plot(history_df['epoch'], history_df['valid_macro_f1'], marker='o', label='val macro F1')
axes[1, 0].axhline(
    V5_REFERENCE_FOLD0_MACRO_F1,
    color='black',
    linestyle='--',
    label='v5 full-run fold 0 reference (not protocol-matched)',
)
axes[1, 0].axhline(GO_THRESHOLD, color='green', linestyle=':', label='project go gate')
axes[1, 0].set_xlabel('epoch')
axes[1, 0].set_ylabel('macro F1')
axes[1, 0].set_title('Probe validation history')
axes[1, 0].legend()

axes[1, 1].hist(confidence[prediction_table['correct'].to_numpy()], bins=20, alpha=0.7, label='correct')
axes[1, 1].hist(confidence[~prediction_table['correct'].to_numpy()], bins=20, alpha=0.7, label='error')
axes[1, 1].set_xlabel('confidence')
axes[1, 1].set_ylabel('rows')
axes[1, 1].set_title('Confidence by correctness')
axes[1, 1].legend()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'fold0_diagnostics.png', dpi=170, bbox_inches='tight')
plt.show()

shown_errors = high_confidence_errors.head(12)
if len(shown_errors):
    columns = 4
    rows = math.ceil(len(shown_errors) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(13, 3.2 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis('off')
    for axis, row in zip(axes.flat, shown_errors.itertuples(index=False)):
        axis.imshow(load_rgb_image(row.effective_image_path))
        axis.set_title(
            f'true={CLASS_NAMES[row.label]}\npred={CLASS_NAMES[row.prediction]} '
            f'p={row.confidence:.3f}, m={row.margin:.3f}',
            fontsize=8,
        )
        axis.axis('off')
else:
    fig, axis = plt.subplots(figsize=(7, 2))
    axis.text(0.5, 0.5, 'Tidak ada error fold 0 untuk ditampilkan.', ha='center', va='center')
    axis.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'fold0_error_examples.png', dpi=150, bbox_inches='tight')
plt.show()

reference_delta_vs_v5_fullrun_fold0 = float(
    fold0_macro_f1 - V5_REFERENCE_FOLD0_MACRO_F1
)
if fold0_macro_f1 >= GO_THRESHOLD:
    decision_category = 'go_future_full5_subject_to_runtime_and_error_review'
    recommendation = (
        'V6 memenuhi project go gate untuk kandidat future full five-fold, dengan syarat '
        'runtime feasible dan review error tidak menunjukkan masalah penting. Ini bukan bukti '
        'kausal bahwa H+ mengalahkan ViT-L karena referensi v5 tidak protocol-matched.'
    )
elif fold0_macro_f1 >= CONDITIONAL_THRESHOLD:
    decision_category = 'conditional_review_before_any_additional_fold'
    recommendation = (
        'Conditional project gate: tinjau per-class, Recyclable/Organic, log loss, confidence, dan error. '
        'Corrected/regressed versus v5 memerlukan prediksi fold-0 v5 yang aligned dan tidak tersedia di sini.'
    )
else:
    decision_category = 'stop_hplus_main_path_prefer_future_partial_ft_vitl'
    recommendation = (
        'Hentikan scaling H+ sebagai jalur utama berikutnya; prioritaskan eksperimen terkontrol '
        'partial fine-tuning ViT-L pada pekerjaan terpisah.'
    )

feature_runtime['oom_occurred'] = False
feature_runtime['failure_count_in_completed_run'] = 0
feature_runtime['peak_vram_by_gpu_gib'] = [
    {
        'device_id': row['device_id'],
        'peak_allocated_gib': row['peak_allocated_gib'],
        'peak_reserved_gib': row['peak_reserved_gib'],
    }
    for row in feature_runtime['gpu_memory_at_completion']
]
elapsed_at_decision = time.perf_counter() - NOTEBOOK_STARTED_AT
operationally_feasible = bool(
    PREFLIGHT_PASSED
    and preflight_report['pass']
    and feature_runtime['completed']
    and cuda_count == 2
    and all('T4' in row['name'].upper() for row in GPU_RUNTIME_INFO)
    and np.isfinite(elapsed_at_decision)
)
operational_evidence = {
    'completed_in_current_kaggle_session': True,
    'gpu_devices': GPU_RUNTIME_INFO,
    'preflight_report': preflight_report,
    'feature_runtime': feature_runtime,
    'probe_training_seconds': PROBE_TRAINING_SECONDS,
    'elapsed_seconds_at_decision': elapsed_at_decision,
    'feasibility_rule': (
        'feasible iff the locked pipeline reaches this decision on exactly T4 x2 without '
        'OOM/failure; no ungrounded wall-clock threshold is invented'
    ),
}
baseline_comparison = pd.DataFrame([
    {
        'experiment': 'v5 ViT-L frozen raw full-run fold 0 reference',
        'macro_f1': V5_REFERENCE_FOLD0_MACRO_F1,
        'reference_delta_vs_v5_fullrun_fold0': 0.0,
        'meets_project_go_gate': None,
        'metric_role': 'project_reference_from_completed_v5_fullrun_not_protocol_matched',
        'probe_training_views': 'orig+hflip',
        'head_seeds': '42+314',
        'max_epochs_budget': 15,
    },
    {
        'experiment': 'v6 ViT-H+ frozen raw compute-efficient fold 0 screen',
        'macro_f1': fold0_macro_f1,
        'reference_delta_vs_v5_fullrun_fold0': reference_delta_vs_v5_fullrun_fold0,
        'meets_project_go_gate': bool(fold0_macro_f1 >= GO_THRESHOLD),
        'metric_role': 'runtime_v6_compute_efficient_fold0_screen',
        'probe_training_views': 'orig',
        'head_seeds': '42',
        'max_epochs_budget': MAX_EPOCHS,
    },
])
write_csv_atomic(baseline_comparison, ARTIFACT_DIR / 'baseline_comparison.csv')

fold0_metrics = {
    'schema_version': 3,
    'revision_marker': V6_REVISION_MARKER,
    'metric_scope': 'fold0_original_view_train_only_compute_efficient_screen',
    'comparison_semantics': 'v5_reference_is_not_protocol_matched_and_delta_is_not_causal',
    'seed': SEED,
    'feature_view': FEATURE_VIEW,
    'max_epochs_budget': MAX_EPOCHS,
    'evaluated_rows': len(valid_indices),
    'macro_f1': float(fold0_macro_f1),
    'accuracy': float(fold0_accuracy),
    'multiclass_log_loss': float(fold0_log_loss),
    'classification_report': report,
    'confusion_matrix_raw': matrix.tolist(),
    'confusion_matrix_row_normalized': normalized_matrix.tolist(),
    'prediction_distribution': prediction_distribution,
    'confidence_summary': distribution_summary(confidence),
    'top_two_margin_summary': distribution_summary(margin),
    'pairwise_confusions': pairwise_confusions,
    'high_confidence_error_count': len(high_confidence_errors),
    'best_epoch': best_epoch,
    'probe_training_seconds': PROBE_TRAINING_SECONDS,
    'feature_runtime': feature_runtime,
    'preflight_report': preflight_report,
    'parameter_counts': {
        'backbone_total': BACKBONE_PARAMETER_COUNT,
        'backbone_trainable': BACKBONE_TRAINABLE_PARAMETER_COUNT,
        'probe_total': PROBE_PARAMETER_COUNT,
        'probe_trainable': PROBE_TRAINABLE_PARAMETER_COUNT,
    },
    'v5_fullrun_fold0_reference_macro_f1': V5_REFERENCE_FOLD0_MACRO_F1,
    'reference_delta_vs_v5_fullrun_fold0': reference_delta_vs_v5_fullrun_fold0,
}
write_json_atomic(fold0_metrics, ARTIFACT_DIR / 'fold0_metrics.json')

screen_decision = {
    'schema_version': 3,
    'revision_marker': V6_REVISION_MARKER,
    'metric_scope': 'fold0_original_view_train_only_compute_efficient_screen',
    'decision_rule_role': 'project_gate_not_causal_backbone_proof',
    'seed': SEED,
    'feature_view': FEATURE_VIEW,
    'max_epochs_budget': MAX_EPOCHS,
    'v5_fullrun_fold0_reference_macro_f1': V5_REFERENCE_FOLD0_MACRO_F1,
    'v6_fold0_macro_f1': float(fold0_macro_f1),
    'reference_delta_vs_v5_fullrun_fold0': reference_delta_vs_v5_fullrun_fold0,
    'comparability_limitation': (
        'The numerical reference delta is not a pure backbone effect: completed v5 used '
        'orig+hflip probe-training views and seeds 42+314, while v6 uses orig and seed 42.'
    ),
    'thresholds': {
        'go_future_full5_minimum': GO_THRESHOLD,
        'conditional_minimum': CONDITIONAL_THRESHOLD,
    },
    'category': decision_category,
    'recommendation': recommendation,
    'project_go_gate_met': bool(fold0_macro_f1 >= GO_THRESHOLD),
    'operationally_feasible_on_observed_t4_x2_run': operationally_feasible,
    'operational_evidence': operational_evidence,
    'automatic_follow_on_started': False,
}
write_json_atomic(screen_decision, ARTIFACT_DIR / 'screen_decision.json')

artifact_paths = {
    'preflight_report': str(PREFLIGHT_REPORT_PATH),
    'fold_assignments': str(ARTIFACT_DIR / 'fold_assignments.csv'),
    'data_and_fold_signature': str(ARTIFACT_DIR / 'data_and_fold_signature.json'),
    'feature_cache_shards': str(CACHE_NAMESPACE),
    'feature_cache_manifest': str(ARTIFACT_DIR / 'feature_cache_manifest.json'),
    'head_history': str(ARTIFACT_DIR / 'head_history_fold0_seed42.csv'),
    'head_checkpoint': str(CHECKPOINT_PATH),
    'fold0_predictions': str(ARTIFACT_DIR / 'fold0_predictions.csv'),
    'fold0_metrics': str(ARTIFACT_DIR / 'fold0_metrics.json'),
    'classification_report': str(ARTIFACT_DIR / 'classification_report.csv'),
    'confusion_matrix': str(ARTIFACT_DIR / 'confusion_matrix.csv'),
    'high_confidence_errors': str(ARTIFACT_DIR / 'high_confidence_errors.csv'),
    'diagnostics_figure': str(ARTIFACT_DIR / 'fold0_diagnostics.png'),
    'error_examples_figure': str(ARTIFACT_DIR / 'fold0_error_examples.png'),
    'screen_decision': str(ARTIFACT_DIR / 'screen_decision.json'),
    'run_manifest': str(ARTIFACT_DIR / 'run_manifest.json'),
}
run_manifest = {
    'schema_version': 3,
    'revision_marker': V6_REVISION_MARKER,
    'experiment_name': EXPERIMENT_NAME,
    'purpose': EXPERIMENT_PURPOSE,
    'source_derived_assumptions': {
        'competition_metric': 'macro_f1',
        'class_names': list(CLASS_NAMES),
        'raw_class_counts': {str(key): value for key, value in EXPECTED_RAW_COUNTS.items()},
        'canonical_data': 'official raw train labels',
        'v5_fullrun_fold0_reference_macro_f1': V5_REFERENCE_FOLD0_MACRO_F1,
        'v5_reference_protocol': {
            'probe_training_views': ['orig', 'hflip'],
            'head_seeds': [42, 314],
            'max_epochs': 15,
            'reference_role': 'project_reference_from_completed_v5_fullrun_not_protocol_matched',
            'full_oof_original_only_macro_f1': 0.986149,
            'full_oof_orig_plus_hflip_prediction_average_macro_f1': 0.986239,
            'selected_final_inference_views': ['orig'],
            'hflip_interpretation': (
                'The 0.000090 gain measures inference probability averaging, not the value '
                'of hflip as a probe-training feature augmentation.'
            ),
        },
        'v6_screen_protocol': {
            'probe_training_views': ['orig'],
            'head_seeds': [42],
            'validation_fold': 0,
            'max_epochs': MAX_EPOCHS,
        },
        'screen_design': (
            'compute-efficient H+ screen preserving raw data, fold, representation, head, '
            'resolution, and original-view evaluation; not a one-variable causal comparison'
        ),
        'comparability_limitation': (
            'The numerical reference delta must not be interpreted as a pure backbone effect '
            'because completed v5 used additional probe-training view and seed ensembling.'
        ),
        'epoch_budget_source': (
            'Matched to the latest completed v5 full-probe budget: FULL_EPOCHS=15.'
        ),
    },
    'runtime_observed_facts': {
        'model_id': MODEL_ID,
        'revision_marker': V6_REVISION_MARKER,
        'immutable_model_revision': RESOLVED_REVISION,
        'hidden_size': HIDDEN_SIZE,
        'patch_size': PATCH_SIZE,
        'num_register_tokens': NUM_REGISTER_TOKENS,
        'patch_token_count': EXPECTED_PATCH_TOKENS,
        'sequence_token_count': EXPECTED_SEQUENCE_TOKENS,
        'processor_class': PREPROCESS_CONFIG['processor_class'],
        'preprocess_signature': PREPROCESS_SIGNATURE,
        'raw_row_count': len(run_df),
        'raw_data_sha256': RAW_DATA_SHA256,
        'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256,
        'validation_fold': VALIDATION_FOLD,
        'seed': SEED,
        'image_size': IMAGE_SIZE,
        'feature_view': FEATURE_VIEW,
        'max_probe_epochs': MAX_EPOCHS,
        'pooling': POOLING_REPRESENTATION,
        'backbone_compute_dtype': BACKBONE_COMPUTE_DTYPE_NAME,
        'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
        'attention_implementation': ATTENTION_IMPLEMENTATION,
        'gpu_strategy': 'torch.nn.DataParallel cuda:0,cuda:1 total batch 2',
        'gpu_devices': GPU_RUNTIME_INFO,
        'preflight_report': preflight_report,
        'feature_runtime': feature_runtime,
        'probe_training_seconds': PROBE_TRAINING_SECONDS,
        'notebook_elapsed_seconds_at_manifest': time.perf_counter() - NOTEBOOK_STARTED_AT,
        'torch_version': torch.__version__,
        'transformers_version': transformers.__version__,
        'parameter_counts': fold0_metrics['parameter_counts'],
    },
    'processor_configuration': PREPROCESS_CONFIG,
    'fold_definition': data_and_fold_signature['fold_definition'],
    'probe_recipe': expected_checkpoint_metadata,
    'baseline_comparison': baseline_comparison.to_dict(orient='records'),
    'decision_gate': screen_decision,
    'artifacts': artifact_paths,
    'compliance_flags': {
        'test_content_accessed': False,
        'test_inference': False,
        'submission_created': False,
        'external_labels': False,
        'relabeling': False,
        'duplicate_removal': False,
        'ensemble': False,
        'partial_fine_tuning': False,
        'automatic_upload': False,
        'automatic_follow_on_experiment': False,
    },
}
write_json_atomic(run_manifest, ARTIFACT_DIR / 'run_manifest.json')

required_artifacts = [Path(path) for path in artifact_paths.values()]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise RuntimeError(f'Artifact contract belum lengkap: {missing_artifacts}.')

display(baseline_comparison)
display(report_frame)
print('Prediction distribution:', prediction_distribution)
print('Recyclable↔Organic:', pairwise_confusions)
print(
    'reference_delta_vs_v5_fullrun_fold0 (not protocol-matched, not causal):',
    reference_delta_vs_v5_fullrun_fold0,
)
print('Decision:', decision_category)
print('Recommendation:', recommendation)
print('Operationally feasible on completed T4 x2 run:', operationally_feasible)
print('All required artifacts:', ARTIFACT_DIR)


## 9. Interpretasi dan langkah berikutnya (tidak dijalankan otomatis)

Gunakan `screen_decision.json`, bukan leaderboard, untuk keputusan. Referensi v5 tidak protocol-matched, sehingga `reference_delta_vs_v5_fullrun_fold0` bukan efek kausal backbone. Jika kategori **go**, full five-fold H+ tetap pekerjaan Kaggle terpisah dan harus meninjau runtime/error lebih dahulu. Jika **conditional**, audit per-class, Recyclable↔Organic, log loss, confidence, serta contoh error sebelum menambah fold. Jika **stop**, kandidat berikutnya adalah partial fine-tuning ViT-L yang dikontrol—bukan perubahan tersembunyi di v6 ini.

Ide resolusi 448/512, partial fine-tuning, atau backbone lain adalah eksperimen masa depan; semuanya sengaja tidak diimplementasikan di sini.

### Checklist yang harus diverifikasi oleh run Kaggle

- [ ] Raw official train 26.527 dan fold seed 42 lolos audit.
- [ ] Preflight official processor + frozen H+ FP32 lolos pada tepat T4 ×2 sebelum safe-copy.
- [ ] Cache FP16 hanya ditulis setelah finite/range checks FP32.
- [ ] CLS + mean true-patch mengecualikan register token secara dinamis.
- [ ] Hanya fold 0, satu head seed 42, original-only, dan maksimal 15 epoch dijalankan.
- [ ] Tidak ada test inference, submission, ensemble, relabeling, atau upload otomatis.
- [ ] Tidak ada eksperimen lanjutan yang dimulai otomatis.